# Data Quality — Missing Value Validation

## 1. Introduction and Objectives

This notebook implements **Task 1 only: missing-value validation** across all nine Olist source datasets. It builds on `01_data_understanding.ipynb` by investigating *why* values are absent, whether the absence is expected or suspicious, and how missingness affects the two purchase-time ML objectives.

The authoritative prediction boundary is order placement (`order_purchase_timestamp`). Later lifecycle timestamps may be used to assess label availability, but not as model inputs. In particular:

- **Classification:** `is_late` requires valid actual and estimated customer-delivery timestamps.
- **Regression:** `delivery_days` requires valid purchase and actual customer-delivery timestamps.
- Reviews and post-purchase status/timestamps are unavailable at prediction time and are excluded from future features.

This is a read-only investigation. It does not clean, impute, remove, overwrite, or model any records. Duplicate validation, broad type checks, outliers, referential integrity, and comprehensive timestamp consistency belong to later tasks.

## 2. Load All Nine Datasets

**What are we checking?** That every expected source file is available and can be loaded with a stable, project-relative path.

**Why does it matter?** Missingness conclusions must come from the current CSV snapshot, not copied counts or hidden notebook state.

**What should we learn?** The shape table establishes the denominator used by every percentage below. Only the three lifecycle fields needed for order-missingness analysis are parsed as datetimes; the source files remain untouched.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 100)
pd.set_option("display.float_format", "{:.4f}".format)


def find_project_root(start: Path) -> Path:
    '''Find the repository root from either the project or notebooks directory.'''
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "data" / "raw").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the project data/raw directory.")


PROJECT_ROOT = find_project_root(Path.cwd())
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"

DATASET_FILES = {
    "Orders": "olist_orders_dataset.csv",
    "Order Items": "olist_order_items_dataset.csv",
    "Products": "olist_products_dataset.csv",
    "Customers": "olist_customers_dataset.csv",
    "Sellers": "olist_sellers_dataset.csv",
    "Payments": "olist_order_payments_dataset.csv",
    "Reviews": "olist_order_reviews_dataset.csv",
    "Geolocation": "olist_geolocation_dataset.csv",
    "Category Translation": "product_category_name_translation.csv",
}


def load_source_csv(file_name: str) -> pd.DataFrame:
    '''Load one immutable source CSV and handle a possible UTF-8 BOM.'''
    file_path = RAW_DATA_DIR / file_name
    if not file_path.is_file():
        raise FileNotFoundError(f"Required source file not found: {file_path}")
    return pd.read_csv(file_path, encoding="utf-8-sig")


datasets = {
    dataset_name: load_source_csv(file_name)
    for dataset_name, file_name in DATASET_FILES.items()
}

orders = datasets["Orders"]
order_items = datasets["Order Items"]
products = datasets["Products"]
customers = datasets["Customers"]
sellers = datasets["Sellers"]
payments = datasets["Payments"]
reviews = datasets["Reviews"]
geolocation = datasets["Geolocation"]
category_translation = datasets["Category Translation"]

ORDER_DATETIME_COLUMNS = [
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
]
for column in ORDER_DATETIME_COLUMNS:
    orders[column] = pd.to_datetime(orders[column], errors="coerce")

dataset_shapes = pd.DataFrame(
    [
        {
            "Dataset": dataset_name,
            "Source CSV": DATASET_FILES[dataset_name],
            "Rows": len(dataframe),
            "Columns": dataframe.shape[1],
        }
        for dataset_name, dataframe in datasets.items()
    ]
)
display(dataset_shapes)

,Dataset,Source CSV,Rows,Columns
0,Orders,olist_orders_dataset.csv,99441,8
1,Order Items,olist_order_items_dataset.csv,112650,7
2,Products,olist_products_dataset.csv,32951,9
3,Customers,olist_customers_dataset.csv,99441,5
4,Sellers,olist_sellers_dataset.csv,3095,4
5,Payments,olist_order_payments_dataset.csv,103886,5
6,Reviews,olist_order_reviews_dataset.csv,99224,7
7,Geolocation,olist_geolocation_dataset.csv,1000163,5
8,Category Translation,product_category_name_translation.csv,71,2


## 3. Missing Values Overview

**What are we checking?** Pandas `NaN` counts and percentages for every column, plus the number of rows containing at least one missing value.

**Why does it matter?** A missing-cell count and an affected-row count answer different questions. One row may contain several missing cells but should count only once in the row-level measure.

**What should we learn?** Percentages use each dataset's row count as the denominator. A nonzero result is an investigation prompt—not automatic proof of a defect—because a field can be optional, not yet applicable in a business process, or genuinely absent.

In [2]:
def missing_value_profile(
    dataset_name: str, dataframe: pd.DataFrame
) -> tuple[dict, pd.DataFrame]:
    '''Return dataset-level and per-column Pandas-NaN summaries.

    Percentages use the dataset row count. The helper reports evidence only and
    deliberately performs no filling, filtering, or source-data mutation.
    '''
    row_count = len(dataframe)
    missing_counts = dataframe.isna().sum()
    rows_with_missing = int(dataframe.isna().any(axis=1).sum())

    dataset_summary = {
        "Dataset": dataset_name,
        "Total Rows": row_count,
        "Column Names": ", ".join(dataframe.columns),
        "Rows With Missing": rows_with_missing,
        "Rows With Missing (%)": (
            rows_with_missing / row_count * 100 if row_count else 0.0
        ),
    }
    column_summary = pd.DataFrame(
        {
            "Dataset": dataset_name,
            "Column": dataframe.columns,
            "Missing Count": missing_counts.to_numpy(),
            "Missing Percentage": (
                missing_counts.to_numpy() / row_count * 100
                if row_count
                else 0.0
            ),
        }
    )
    return dataset_summary, column_summary


dataset_missing_rows = []
column_missing_tables = []
for dataset_name, dataframe in datasets.items():
    dataset_summary, column_summary = missing_value_profile(
        dataset_name, dataframe
    )
    dataset_missing_rows.append(dataset_summary)
    column_missing_tables.append(column_summary)

dataset_missing_overview = pd.DataFrame(dataset_missing_rows)
column_missing_overview = pd.concat(column_missing_tables, ignore_index=True)

display(dataset_missing_overview)
display(column_missing_overview)

,Dataset,Total Rows,Column Names,Rows With Missing,Rows With Missing (%)
0,Orders,99441,"order_id, customer_id, order_status, order_purchase_timestamp, order_approved_at, order_delivere...",2980,2.9968
1,Order Items,112650,"order_id, order_item_id, product_id, seller_id, shipping_limit_date, price, freight_value",0,0.0000
2,Products,32951,"product_id, product_category_name, product_name_lenght, product_description_lenght, product_phot...",611,1.8543
3,Customers,99441,"customer_id, customer_unique_id, customer_zip_code_prefix, customer_city, customer_state",0,0.0000
4,Sellers,3095,"seller_id, seller_zip_code_prefix, seller_city, seller_state",0,0.0000
5,Payments,103886,"order_id, payment_sequential, payment_type, payment_installments, payment_value",0,0.0000
6,Reviews,99224,"review_id, order_id, review_score, review_comment_title, review_comment_message, review_creation...",89385,90.0841
7,Geolocation,1000163,"geolocation_zip_code_prefix, geolocation_lat, geolocation_lng, geolocation_city, geolocation_state",0,0.0000
8,Category Translation,71,"product_category_name, product_category_name_english",0,0.0000


,Dataset,Column,Missing Count,Missing Percentage
0,Orders,order_id,0,0.0000
1,Orders,customer_id,0,0.0000
2,Orders,order_status,0,0.0000
3,Orders,order_purchase_timestamp,0,0.0000
4,Orders,order_approved_at,160,0.1609
5,Orders,order_delivered_carrier_date,1783,1.7930
6,Orders,order_delivered_customer_date,2965,2.9817
7,Orders,order_estimated_delivery_date,0,0.0000
8,Order Items,order_id,0,0.0000
9,Order Items,order_item_id,0,0.0000


**Interpretation guidance:** Compare `Rows With Missing` with the per-column counts. Orders and products can have related fields absent together; reviews can have one or both optional text fields absent. Datasets with zero Pandas `NaN` still need a hidden-missingness check for blank strings and placeholder tokens.

## 4. Orders — Missing Value Validation

**What are we checking?** Missing approval, carrier-handoff, and actual customer-delivery timestamps by `order_status`, including delivered orders missing their actual-delivery timestamp.

**Why does it matter?** Lifecycle events may legitimately be absent when an order has not reached that stage. However, `order_delivered_customer_date` is required to observe both ML targets, so a delivered-status row without it is suspicious and label-ineligible until investigated.

**What should we learn?** The status-conditioned tables separate expected process-related missingness from contradictory combinations. Status itself is post-purchase information and is used here only for retrospective validation—not as a purchase-time feature.

In [3]:
ORDER_MISSING_COLUMNS = [
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
]

orders_missing_summary = pd.DataFrame(
    {
        "Column": ORDER_MISSING_COLUMNS,
        "Missing Count": orders[ORDER_MISSING_COLUMNS].isna().sum().to_numpy(),
        "Missing Percentage": (
            orders[ORDER_MISSING_COLUMNS].isna().mean().mul(100).to_numpy()
        ),
    }
)
display(orders_missing_summary)

order_status_missing_counts = pd.concat(
    {
        column: pd.crosstab(
            orders["order_status"],
            orders[column].isna(),
        ).rename(columns={False: "Present", True: "Missing"})
        for column in ORDER_MISSING_COLUMNS
    },
    names=["Column", "order_status"],
).astype("Int64")
display(order_status_missing_counts)

,Column,Missing Count,Missing Percentage
0,order_approved_at,160,0.1609
1,order_delivered_carrier_date,1783,1.7930
2,order_delivered_customer_date,2965,2.9817


Present  Missing
Column                        order_status                  
order_approved_at             approved            2        0
                              canceled          484      141
                              created             0        5
                              delivered       96464       14
                              invoiced          314        0
                              processing        301        0
                              shipped          1107        0
                              unavailable       609        0
order_delivered_carrier_date  approved            0        2
                              canceled           75      550
                              created             0        5
                              delivered       96476        2
                              invoiced            0      314
                              processing          0      301
                              shipped          1107        0
                              unavailable         0      609
order_delivered_customer_date approved            0        2
                              canceled            6      619
                              created             0        5
                              delivered       96470        8
                              invoiced            0      314
                              processing          0      301
                              shipped             0     1107
                              unavailable         0      609

In [4]:
order_suspicion_masks = {
    "Delivered but approval timestamp missing": (
        orders["order_status"].eq("delivered")
        & orders["order_approved_at"].isna()
    ),
    "Delivered but carrier timestamp missing": (
        orders["order_status"].eq("delivered")
        & orders["order_delivered_carrier_date"].isna()
    ),
    "Delivered but customer-delivery timestamp missing": (
        orders["order_status"].eq("delivered")
        & orders["order_delivered_customer_date"].isna()
    ),
    "Non-delivered status but customer-delivery timestamp present": (
        orders["order_status"].ne("delivered")
        & orders["order_delivered_customer_date"].notna()
    ),
}

order_suspicious_summary = pd.DataFrame(
    [
        {"Condition": condition, "Record Count": int(mask.sum())}
        for condition, mask in order_suspicion_masks.items()
    ]
)
display(order_suspicious_summary)

suspicious_order_mask = pd.concat(
    order_suspicion_masks.values(), axis=1
).any(axis=1)
suspicious_order_columns = [
    "order_id",
    "order_status",
    "order_purchase_timestamp",
    *ORDER_MISSING_COLUMNS,
    "order_estimated_delivery_date",
]
display(orders.loc[suspicious_order_mask, suspicious_order_columns].head(10))

,Condition,Record Count
0,Delivered but approval timestamp missing,14
1,Delivered but carrier timestamp missing,2
2,Delivered but customer-delivery timestamp missing,8
3,Non-delivered status but customer-delivery timestamp present,6


,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
2921,1950d777989f6a877539f53795b4c3c3,canceled,2018-02-19 19:48:52,2018-02-19 20:56:05,2018-02-20 19:57:13,2018-03-21 22:03:51,2018-03-09 00:00:00
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,delivered,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,NaT,2017-12-18 00:00:00
5323,e04abd8149ef81b95221e88f6ed9ab6a,delivered,2017-02-18 14:40:00,NaT,2017-02-23 12:04:47,2017-03-01 13:25:33,2017-03-17 00:00:00
8791,dabf2b0e35b423f94618bf965fcb7514,canceled,2016-10-09 00:56:52,2016-10-09 13:36:58,2016-10-13 13:36:59,2016-10-16 14:36:59,2016-11-30 00:00:00
16567,8a9adc69528e1001fc68dd0aaebbb54a,delivered,2017-02-18 12:45:31,NaT,2017-02-23 09:01:52,2017-03-02 10:05:06,2017-03-21 00:00:00
19031,7013bcfc1c97fe719a7b5e05e61c12db,delivered,2017-02-18 13:29:47,NaT,2017-02-22 16:25:25,2017-03-01 08:07:38,2017-03-17 00:00:00
20618,f5dd62b788049ad9fc0526e3ad11a097,delivered,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,NaT,2018-07-16 00:00:00
22663,5cf925b116421afa85ee25e99b4c34fb,delivered,2017-02-18 16:48:35,NaT,2017-02-22 11:23:10,2017-03-09 07:28:47,2017-03-31 00:00:00
23156,12a95a3c06dbaec84bcfb0e2da5d228a,delivered,2017-02-17 13:05:55,NaT,2017-02-22 11:23:11,2017-03-02 11:09:19,2017-03-20 00:00:00
26800,c1d4211b3dae76144deccd6c74144a88,delivered,2017-01-19 12:48:08,NaT,2017-01-25 14:56:50,2017-01-30 18:16:01,2017-03-01 00:00:00


**Interpretation guidance:** Missing lifecycle timestamps among canceled, unavailable, created, invoiced, processing, approved, or shipped orders may reflect an event that never occurred or had not occurred in the extract; those orders are not automatically invalid. Delivered-status contradictions require investigation, and so do the canceled orders that nevertheless carry an actual customer-delivery timestamp: a timestamp-only eligibility rule would treat them as labelable, while a status-based rule would exclude them. That choice is deferred to target construction, but it must be made explicitly. Any row without an observed customer-delivery timestamp cannot receive either target label under the authoritative definitions; it must not be silently labeled on time or assigned a duration. Approval and carrier timestamps are post-purchase and are never candidate predictors here.

## 5. Products — Missing Value Validation

**What are we checking?** Whether descriptive catalog fields and physical measurements are missing individually or as coordinated groups.

**Why does it matter?** Co-missing fields can reveal a systematic catalog-ingestion pattern, while isolated physical missingness may indicate a small number of incomplete product records. Both can affect later feature availability.

**What should we learn?** Evidence about co-missingness should guide future treatment decisions. This notebook does not impute values or approve any feature for modeling.

In [5]:
PRODUCT_METADATA_COLUMNS = [
    "product_category_name",
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty",
]
PRODUCT_PHYSICAL_COLUMNS = [
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm",
]
PRODUCT_INVESTIGATION_COLUMNS = (
    PRODUCT_METADATA_COLUMNS + PRODUCT_PHYSICAL_COLUMNS
)

product_missing_summary = pd.DataFrame(
    {
        "Column": PRODUCT_INVESTIGATION_COLUMNS,
        "Missing Count": products[PRODUCT_INVESTIGATION_COLUMNS]
        .isna()
        .sum()
        .to_numpy(),
        "Missing Percentage": products[PRODUCT_INVESTIGATION_COLUMNS]
        .isna()
        .mean()
        .mul(100)
        .to_numpy(),
    }
)
display(product_missing_summary)

,Column,Missing Count,Missing Percentage
0,product_category_name,610,1.8512
1,product_name_lenght,610,1.8512
2,product_description_lenght,610,1.8512
3,product_photos_qty,610,1.8512
4,product_weight_g,2,0.0061
5,product_length_cm,2,0.0061
6,product_height_cm,2,0.0061
7,product_width_cm,2,0.0061


In [6]:
product_missing_flags = products[PRODUCT_INVESTIGATION_COLUMNS].isna()
product_pattern_counts = (
    product_missing_flags.value_counts()
    .rename("Product Count")
    .reset_index()
    .sort_values("Product Count", ascending=False)
)
display(product_pattern_counts.head(10))

metadata_missing_count = product_missing_flags[PRODUCT_METADATA_COLUMNS].sum(axis=1)
physical_missing_count = product_missing_flags[PRODUCT_PHYSICAL_COLUMNS].sum(axis=1)

product_group_patterns = pd.DataFrame(
    [
        {
            "Pattern": "All four descriptive metadata fields missing",
            "Product Count": int(
                product_missing_flags[PRODUCT_METADATA_COLUMNS].all(axis=1).sum()
            ),
        },
        {
            "Pattern": "Some, but not all, descriptive fields missing",
            "Product Count": int(
                ((metadata_missing_count > 0) & (metadata_missing_count < 4)).sum()
            ),
        },
        {
            "Pattern": "All four physical attributes missing",
            "Product Count": int(
                product_missing_flags[PRODUCT_PHYSICAL_COLUMNS].all(axis=1).sum()
            ),
        },
        {
            "Pattern": "Some, but not all, physical attributes missing",
            "Product Count": int(
                ((physical_missing_count > 0) & (physical_missing_count < 4)).sum()
            ),
        },
        {
            "Pattern": "Both metadata and physical groups affected",
            "Product Count": int(
                ((metadata_missing_count > 0) & (physical_missing_count > 0)).sum()
            ),
        },
    ]
)
display(product_group_patterns)

product_missing_examples = products.loc[
    product_missing_flags.any(axis=1),
    ["product_id", *PRODUCT_INVESTIGATION_COLUMNS],
].head(10)
display(product_missing_examples)

,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,Product Count
0,False,False,False,False,False,False,False,False,32340
1,True,True,True,True,False,False,False,False,609
2,False,False,False,False,True,True,True,True,1
3,True,True,True,True,True,True,True,True,1


,Pattern,Product Count
0,All four descriptive metadata fields missing,610
1,"Some, but not all, descriptive fields missing",0
2,All four physical attributes missing,2
3,"Some, but not all, physical attributes missing",0
4,Both metadata and physical groups affected,1


,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
105,a41e356c76fab66334f36de622ecbd3a,NaN,NaN,NaN,NaN,650.0000,17.0000,14.0000,12.0000
128,d8dee61c2034d6d075997acef1870e9b,NaN,NaN,NaN,NaN,300.0000,16.0000,7.0000,20.0000
145,56139431d72cd51f19eb9f7dae4d1617,NaN,NaN,NaN,NaN,200.0000,20.0000,20.0000,20.0000
154,46b48281eb6d663ced748f324108c733,NaN,NaN,NaN,NaN,18500.0000,41.0000,30.0000,41.0000
197,5fb61f482620cb672f5e586bb132eae9,NaN,NaN,NaN,NaN,300.0000,35.0000,7.0000,12.0000
244,e10758160da97891c2fdcbc35f0f031d,NaN,NaN,NaN,NaN,2200.0000,16.0000,2.0000,11.0000
294,39e3b9b12cd0bf8ee681bbc1c130feb5,NaN,NaN,NaN,NaN,300.0000,16.0000,7.0000,11.0000
299,794de06c32a626a5692ff50e4985d36f,NaN,NaN,NaN,NaN,300.0000,18.0000,8.0000,14.0000
347,7af3e2da474486a3519b0cba9dea8ad9,NaN,NaN,NaN,NaN,200.0000,22.0000,14.0000,14.0000
428,629beb8e7317703dcc5f35b5463fd20e,NaN,NaN,NaN,NaN,1400.0000,25.0000,25.0000,25.0000


**Interpretation guidance:** A dominant all-or-none pattern across category/name-length/description-length/photo fields is evidence of systematic descriptive-metadata absence rather than unrelated random cells. Compare that with the physical group: very few products missing every measurement indicates a separate, isolated issue. These fields may be purchase-time feature candidates only after serving availability is established; later preprocessing must preserve missingness information and learn any treatment from training data only.

## 6. Reviews — Missing Value Validation

**What are we checking?** Optional review title/message absence overall, by review score, and across text-presence combinations.

**Why does it matter?** Sparse optional text is different from a broken required field. Variation by score may also reveal how customers choose to leave written feedback.

**What should we learn?** Reviews are useful for retrospective experience analysis, but they occur after purchase and must not be used as purchase-time predictive features.

In [7]:
REVIEW_TEXT_COLUMNS = ["review_comment_title", "review_comment_message"]

review_missing_summary = pd.DataFrame(
    {
        "Column": REVIEW_TEXT_COLUMNS,
        "Missing Count": reviews[REVIEW_TEXT_COLUMNS].isna().sum().to_numpy(),
        "Missing Percentage": (
            reviews[REVIEW_TEXT_COLUMNS].isna().mean().mul(100).to_numpy()
        ),
    }
)
display(review_missing_summary)

review_missing_by_score = (
    reviews.groupby("review_score")[REVIEW_TEXT_COLUMNS]
    .agg(["count", lambda series: series.isna().sum(), lambda series: series.isna().mean() * 100])
)
review_missing_by_score.columns = [
    f"{column}_{metric}"
    for column, metric in review_missing_by_score.columns
]
review_missing_by_score = review_missing_by_score.rename(
    columns=lambda name: name.replace("_<lambda_0>", "_missing_count").replace(
        "_<lambda_1>", "_missing_percentage"
    )
)
display(review_missing_by_score)

,Column,Missing Count,Missing Percentage
0,review_comment_title,87656,88.3415
1,review_comment_message,58247,58.7025


,review_comment_title_count,review_comment_title_missing_count,review_comment_title_missing_percentage,review_comment_message_count,review_comment_message_missing_count,review_comment_message_missing_percentage
review_score,,,,,,
1,1873,9551,83.6047,8745,2679,23.4506
2,478,2673,84.8302,2145,1006,31.9264
3,824,7355,89.9254,3557,4622,56.5106
4,1735,17407,90.9362,5976,13166,68.7807
5,6658,50670,88.3861,20554,36774,64.1467


In [8]:
review_text_presence = pd.DataFrame(
    {
        "Title Present": reviews["review_comment_title"].notna(),
        "Message Present": reviews["review_comment_message"].notna(),
    }
)
review_text_combinations = (
    review_text_presence.value_counts()
    .rename("Review Count")
    .reset_index()
    .sort_values("Review Count", ascending=False)
)
review_text_combinations["Percentage"] = (
    review_text_combinations["Review Count"] / len(reviews) * 100
)
display(review_text_combinations)

review_text_by_score = pd.crosstab(
    reviews["review_score"],
    review_text_presence["Message Present"],
    normalize="index",
).rename(columns={False: "Without Message (%)", True: "With Message (%)"}) * 100
display(review_text_by_score)

,Title Present,Message Present,Review Count,Percentage
0,False,False,56518,56.9600
1,False,True,31138,31.3815
2,True,True,9839,9.9159
3,True,False,1729,1.7425


Message Present,Without Message (%),With Message (%)
review_score,,
1,23.4506,76.5494
2,31.9264,68.0736
3,56.5106,43.4894
4,68.7807,31.2193
5,64.1467,35.8533


**Interpretation guidance:** The presence/absence combinations show that a review can be valid without either text field; the numeric score and review timestamps remain populated. Score-level differences are behavioral evidence, not a reason to manufacture comments. All review fields describe post-purchase experience and therefore remain excluded from both purchase-time models.

## 7. Other Six Datasets — Hidden Missing Values

**What are we checking?** Pandas `NaN`, empty strings, whitespace-only strings, and a conservative set of common missing-value tokens in Customers, Sellers, Order Items, Payments, Geolocation, and Category Translation.

**Why does it matter?** CSV text can look populated to Pandas while carrying an empty or placeholder-like value. These categories must remain distinct because their meanings and remedies differ.

**Why read the files a second time?** By default `pd.read_csv` converts empty fields and tokens such as `NA`, `N/A`, `null`, `None`, `nan`, and `n/a` into `NaN` while loading. An empty-string or placeholder check on the already-loaded DataFrame can therefore never find those values: they have already become `NaN`. To see what is literally written in each file, we reload it with `dtype=str` and `keep_default_na=False`, which disables that conversion. The `Converted To NaN` column then shows whether Pandas turned any *non-empty* text into `NaN`.

**What should we learn?** A literal token is only a *suspected* placeholder until interpreted in its column context. For example, a documented business category such as `not_defined` is not automatically equivalent to `NaN`.

In [9]:
# Checked against the raw file text, so tokens that Pandas would normally
# convert to NaN on load (for example "NA" or "null") are still visible here.
COMMON_PLACEHOLDER_TOKENS = {"n/a", "na", "nan", "null", "none", "unknown", "?", "-"}
OTHER_DATASETS = {
    "Customers": customers,
    "Sellers": sellers,
    "Order Items": order_items,
    "Payments": payments,
    "Geolocation": geolocation,
    "Category Translation": category_translation,
}


def load_raw_text_csv(file_name: str) -> pd.DataFrame:
    '''Reload a source CSV as literal text with default NA conversion disabled.

    Every column is read as `str` and `keep_default_na=False` keeps empty fields
    as "" and tokens such as "NA" as text, exposing exactly what the file holds.
    '''
    return pd.read_csv(
        RAW_DATA_DIR / file_name,
        dtype=str,
        keep_default_na=False,
        encoding="utf-8-sig",
    )


def hidden_missing_profile(
    dataset_name: str,
    parsed: pd.DataFrame,
    raw: pd.DataFrame,
    columns: list[str] | None = None,
) -> pd.DataFrame:
    '''Compare each column's Pandas NaN count with its raw-text representation.

    `parsed` is the default-loaded DataFrame and `raw` is the same file from
    `load_raw_text_csv`. Counts are evidence only; no values are changed.
    '''
    rows = []
    for column in columns or list(parsed.columns):
        raw_text = raw[column]
        stripped = raw_text.str.strip()
        parsed_nan = parsed[column].isna()
        rows.append(
            {
                "Dataset": dataset_name,
                "Column": column,
                "Pandas NaN": int(parsed_nan.sum()),
                "Empty String": int(raw_text.eq("").sum()),
                "Whitespace Only": int((stripped.eq("") & raw_text.ne("")).sum()),
                "Suspected Placeholder": int(
                    stripped.str.lower().isin(COMMON_PLACEHOLDER_TOKENS).sum()
                ),
                # Non-empty file text that Pandas silently turned into NaN on load.
                "Converted To NaN": int((parsed_nan & raw_text.ne("")).sum()),
            }
        )
    return pd.DataFrame(rows)


HIDDEN_MISSING_COLUMNS = [
    "Pandas NaN",
    "Empty String",
    "Whitespace Only",
    "Suspected Placeholder",
    "Converted To NaN",
]

hidden_missing_details = pd.concat(
    [
        hidden_missing_profile(
            dataset_name, dataframe, load_raw_text_csv(DATASET_FILES[dataset_name])
        )
        for dataset_name, dataframe in OTHER_DATASETS.items()
    ],
    ignore_index=True,
)
display(hidden_missing_details)

hidden_missing_by_dataset = hidden_missing_details.groupby(
    "Dataset", as_index=False
)[HIDDEN_MISSING_COLUMNS].sum()
display(hidden_missing_by_dataset)

# `not_defined` is a documented payment-type category, so it is reported for
# context but not relabeled as missing by the conservative token scan above.
not_defined_payment_count = int(
    payments["payment_type"].astype("string").str.strip().str.lower().eq("not_defined").sum()
)
pd.Series({"payment_type='not_defined' (context only)": not_defined_payment_count})


,Dataset,Column,Pandas NaN,Empty String,Whitespace Only,Suspected Placeholder,Converted To NaN
0,Customers,customer_id,0,0,0,0,0
1,Customers,customer_unique_id,0,0,0,0,0
2,Customers,customer_zip_code_prefix,0,0,0,0,0
3,Customers,customer_city,0,0,0,0,0
4,Customers,customer_state,0,0,0,0,0
5,Sellers,seller_id,0,0,0,0,0
6,Sellers,seller_zip_code_prefix,0,0,0,0,0
7,Sellers,seller_city,0,0,0,0,0
8,Sellers,seller_state,0,0,0,0,0
9,Order Items,order_id,0,0,0,0,0


,Dataset,Pandas NaN,Empty String,Whitespace Only,Suspected Placeholder,Converted To NaN
0,Category Translation,0,0,0,0,0
1,Customers,0,0,0,0,0
2,Geolocation,0,0,0,0,0
3,Order Items,0,0,0,0,0
4,Payments,0,0,0,0,0
5,Sellers,0,0,0,0,0


payment_type='not_defined' (context only)    3
dtype: int64

**Interpretation guidance:** Keep the four representations separate. `NaN` is Pandas' missing marker; an empty string has zero characters; a whitespace-only value contains characters but no visible content; and a placeholder token is merely suspicious until its semantics are verified. `Empty String` should equal `Pandas NaN` for each column (empty fields are what Pandas reads as `NaN`), and `Converted To NaN` should be zero; a nonzero value would mean real text was hidden by the default loader. No values are normalized or overwritten here.

### Review Text — Hidden Missing Values

**What are we checking?** The same raw-text scan applied to `review_comment_title` and `review_comment_message`, the free-text fields where blank or placeholder input is most likely.

**Why does it matter?** Section 6 treats any non-`NaN` comment as "text present". A comment containing only spaces, `?`, or `-` is not informative text, so those rows slightly understate effective text absence.

**What should we learn?** Whether the Section 6 missing percentages are exact or a lower bound. This remains retrospective analysis only; review data is never a purchase-time feature.

In [10]:
review_raw_text = load_raw_text_csv(DATASET_FILES["Reviews"])
review_hidden_missing = hidden_missing_profile(
    "Reviews", reviews, review_raw_text, REVIEW_TEXT_COLUMNS
)
display(review_hidden_missing)

# Show the literal values behind the whitespace/placeholder counts.
review_hidden_examples = pd.concat(
    {
        column: review_raw_text.loc[
            review_raw_text[column].ne("")
            & (
                review_raw_text[column].str.strip().eq("")
                | review_raw_text[column]
                .str.strip()
                .str.lower()
                .isin(COMMON_PLACEHOLDER_TOKENS)
            ),
            column,
        ]
        .map(repr)
        .value_counts()
        for column in REVIEW_TEXT_COLUMNS
    },
    names=["Column", "Raw Value"],
).rename("Review Count")
display(review_hidden_examples)


,Dataset,Column,Pandas NaN,Empty String,Whitespace Only,Suspected Placeholder,Converted To NaN
0,Reviews,review_comment_title,87656,87656,2,2,0
1,Reviews,review_comment_message,58247,58247,27,7,0


Column                  Raw Value                                 
review_comment_title    ' '                                            2
                        '?'                                            1
                        '-'                                            1
review_comment_message  '\r\n'                                        14
                        ' '                                            9
                        '\r\n\r\n'                                     3
                        '?'                                            3
                        '-'                                            3
                        'N/a'                                          1
                        '\r\n\r\n\r\n\r\n\r\n\r\n\r\n\r\n\r\n\r\n'     1
Name: Review Count, dtype: int64

**Interpretation guidance:** These comments are populated in the CSV but carry no readable content. They are a small share of reviews, so the Section 6 conclusions about score-linked optional text do not change, but the reported message and title missing percentages should be read as a lower bound on *effective* text absence. Nothing is rewritten to `NaN` in this task.

## 8. Missingness Patterns

**What are we checking?** The strongest cross-field patterns from the focused investigations: coordinated product absence, order timestamps by status, and review-message absence by score.

**Why does it matter?** Missingness patterns can reveal business-process stages and data-collection mechanisms that isolated column totals hide.

**What should we learn?** Concise pattern tables are sufficient here; a chart would not add meaningful information beyond these exact counts and percentages.

In [11]:
order_customer_delivery_pattern = pd.crosstab(
    orders["order_status"],
    orders["order_delivered_customer_date"].isna(),
).rename(columns={False: "Delivery Timestamp Present", True: "Delivery Timestamp Missing"})
order_customer_delivery_pattern["Missing Percentage Within Status"] = (
    order_customer_delivery_pattern.get("Delivery Timestamp Missing", 0)
    / order_customer_delivery_pattern.sum(axis=1)
    * 100
)

review_message_pattern = (
    reviews.groupby("review_score")["review_comment_message"]
    .agg(
        Reviews="size",
        Missing=lambda series: series.isna().sum(),
    )
)
review_message_pattern["Missing Percentage"] = (
    review_message_pattern["Missing"] / review_message_pattern["Reviews"] * 100
)

display(product_group_patterns)
display(order_customer_delivery_pattern)
display(review_message_pattern)

,Pattern,Product Count
0,All four descriptive metadata fields missing,610
1,"Some, but not all, descriptive fields missing",0
2,All four physical attributes missing,2
3,"Some, but not all, physical attributes missing",0
4,Both metadata and physical groups affected,1


order_delivered_customer_date,Delivery Timestamp Present,Delivery Timestamp Missing,Missing Percentage Within Status
order_status,,,
approved,0,2,100.0000
canceled,6,619,99.0400
created,0,5,100.0000
delivered,96470,8,0.0083
invoiced,0,314,100.0000
processing,0,301,100.0000
shipped,0,1107,100.0000
unavailable,0,609,100.0000


,Reviews,Missing,Missing Percentage
review_score,,,
1,11424,2679,23.4506
2,3151,1006,31.9264
3,8179,4622,56.5106
4,19142,13166,68.7807
5,57328,36774,64.1467


**Interpretation guidance:** Treat these as mechanisms to investigate, not causal proof. Status-linked timestamp absence is consistent with lifecycle progression; coordinated product metadata absence is consistent with a catalog-record pattern; score-linked review text reflects optional response behavior. None of these observations authorizes deletion or imputation.

## 9. ML Impact Assessment

Missingness has different consequences depending on the field's role and availability at order placement.

| Missingness role | Classification impact | Regression impact | Purchase-time interpretation |
|---|---|---|---|
| Actual customer-delivery timestamp | Required to observe `is_late`; missing means no supervised label | Required to calculate `delivery_days`; missing means no supervised label | Outcome-only; never a feature |
| Estimated delivery timestamp | Required for `is_late`; no missing values are observed in this snapshot | Not required for the duration formula, though potentially useful as a promise feature | Intended purchase-time candidate; serving availability still needs validation |
| Purchase timestamp | Needed for regression target start and chronological splitting; no missing values are observed | Required to calculate `delivery_days` | Prediction boundary and potential calendar-feature source |
| Approval/carrier timestamps | Not needed for the target definition | Not needed for the target definition | Post-purchase; missingness may be process-related, but values are prohibited features |
| Product catalog metadata and measurements | May reduce availability of candidate product features | Same potential feature impact | Candidate features only; treatment must be learned later from training data |
| Review title/message | No target requirement and no permitted feature use | No target requirement and no permitted feature use | Post-purchase leakage; retrospective analysis only |
| Status/date contradiction (canceled orders with an actual delivery timestamp) | Would pass a timestamp-only `is_late` eligibility rule but fail a status-based rule | Same ambiguity for `delivery_days` | Eligibility rule is an open decision for target construction; never a feature |
| Expected lifecycle absence | May explain why an order lacks an observed outcome | May explain why an order lacks an observed outcome | Do not equate an unfinished/canceled process with corrupt data |

The supervised training cohort will be narrower than the deployment population because some historical orders lack observable targets. Eligibility must be assessed explicitly in a later target-construction task; missing outcomes must never be converted silently to negative labels or zero duration.

## 10. Findings and Recommendations

**What are we checking?** A consolidated, evidence-linked decision table for every field or dataset investigated in this task.

**Why does it matter?** Column-level totals can mix legitimate and suspicious cases. Classifications below are therefore conditional where status or field group changes the interpretation.

**What should we learn?** Recommendations are investigative and eligibility-oriented. They do not perform cleaning or prescribe an unvalidated imputation strategy.

In [12]:
def missing_count_and_percentage(
    dataframe: pd.DataFrame, column: str
) -> tuple[int, float]:
    '''Return a column's current missing count and row-based percentage.'''
    count = int(dataframe[column].isna().sum())
    percentage = count / len(dataframe) * 100 if len(dataframe) else 0.0
    return count, percentage


canceled_with_delivery_count = int(
    (
        orders["order_status"].eq("canceled")
        & orders["order_delivered_customer_date"].notna()
    ).sum()
)

findings_rows = []

for column in ORDER_MISSING_COLUMNS:
    count, percentage = missing_count_and_percentage(orders, column)
    delivered_missing = int(
        (orders["order_status"].eq("delivered") & orders[column].isna()).sum()
    )
    if column == "order_delivered_customer_date":
        ml_impact = "Classification and regression target eligibility"
        action = "Assess label eligibility; investigate delivered-status and canceled-with-delivery contradictions"
    else:
        ml_impact = "None as a feature; lifecycle-quality context only"
        action = "Retain for audit; investigate delivered-status contradictions"
    findings_rows.append(
        {
            "Dataset": DATASET_FILES["Orders"],
            "Column": column,
            "Missing Count": count,
            "Missing Percentage": percentage,
            "Percentage Basis": "Rows",
            "Classification": "Expected by some statuses / Needs Investigation when delivered",
            "Evidence": (
                f"Missingness varies by order_status; {delivered_missing} delivered rows are missing this field."
            ),
            "ML Impact": ml_impact,
            "Proposed Action": action,
        }
    )

# A contradiction in the opposite direction: an outcome timestamp exists even
# though the status says the order was canceled. This affects label eligibility.
findings_rows.append(
    {
        "Dataset": DATASET_FILES["Orders"],
        "Column": "order_status vs order_delivered_customer_date",
        "Missing Count": canceled_with_delivery_count,
        "Missing Percentage": canceled_with_delivery_count / len(orders) * 100,
        "Percentage Basis": "Rows",
        "Classification": "Needs Investigation",
        "Evidence": (
            f"{canceled_with_delivery_count} canceled orders have an actual customer-delivery timestamp (count of contradictions, not missing cells)."
        ),
        "ML Impact": "Open target-eligibility decision: timestamp-only vs status-based rule",
        "Proposed Action": "Decide and document the eligibility rule in target construction; report the count",
    }
)

for column in PRODUCT_INVESTIGATION_COLUMNS:
    count, percentage = missing_count_and_percentage(products, column)
    if column in PRODUCT_METADATA_COLUMNS:
        classification = "Needs Investigation"
        evidence = (
            "Descriptive fields show coordinated absence; compare the all-four-missing pattern."
        )
        action = "Retain; document missing indicator and consider later train-only treatment"
    else:
        classification = "Needs Investigation"
        evidence = "All four physical measurements are absent together for a very small product set."
        action = "Investigate source records; consider later train-only treatment"
    findings_rows.append(
        {
            "Dataset": DATASET_FILES["Products"],
            "Column": column,
            "Missing Count": count,
            "Missing Percentage": percentage,
            "Percentage Basis": "Rows",
            "Classification": classification,
            "Evidence": evidence,
            "ML Impact": "Potential predictive feature availability",
            "Proposed Action": action,
        }
    )

for column in REVIEW_TEXT_COLUMNS:
    count, percentage = missing_count_and_percentage(reviews, column)
    findings_rows.append(
        {
            "Dataset": DATASET_FILES["Reviews"],
            "Column": column,
            "Missing Count": count,
            "Missing Percentage": percentage,
            "Percentage Basis": "Rows",
            "Classification": "Expected",
            "Evidence": (
                f"Comment text is optional; valid review-score records exist without it. "
                f"A further {int(review_hidden_missing.set_index('Column').loc[column, ['Whitespace Only', 'Suspected Placeholder']].sum())} "
                "populated values are whitespace-only or placeholder-like."
            ),
            "ML Impact": "None as a feature; retrospective analysis only",
            "Proposed Action": "Retain missingness; exclude all review data from purchase-time features",
        }
    )

for dataset_name, dataframe in OTHER_DATASETS.items():
    profile = hidden_missing_details.loc[
        hidden_missing_details["Dataset"].eq(dataset_name)
    ]
    actual_missing = int(profile["Pandas NaN"].sum())
    # Empty strings are excluded here because they are the same cells Pandas
    # already counts as NaN; only additional hidden representations are added.
    hidden_missing = int(
        profile[["Whitespace Only", "Suspected Placeholder", "Converted To NaN"]]
        .sum()
        .sum()
    )
    findings_rows.append(
        {
            "Dataset": DATASET_FILES[dataset_name],
            "Column": "All columns / suitable text columns",
            "Missing Count": actual_missing,
            "Missing Percentage": (
                actual_missing / dataframe.size * 100 if dataframe.size else 0.0
            ),
            "Percentage Basis": "Cells",
            "Classification": "Expected" if actual_missing == 0 and hidden_missing == 0 else "Needs Investigation",
            "Evidence": (
                f"Column scan found {actual_missing} Pandas NaN cells and {hidden_missing} hidden/suspected placeholder matches."
            ),
            "ML Impact": "Potential feature availability if any missingness is found",
            "Proposed Action": "Retain source values; investigate tokens in column context",
        }
    )

findings = pd.DataFrame(findings_rows)
display(findings)

,Dataset,Column,Missing Count,Missing Percentage,Percentage Basis,Classification,Evidence,ML Impact,Proposed Action
0,olist_orders_dataset.csv,order_approved_at,160,0.1609,Rows,Expected by some statuses / Needs Investigation when delivered,Missingness varies by order_status; 14 delivered rows are missing this field.,None as a feature; lifecycle-quality context only,Retain for audit; investigate delivered-status contradictions
1,olist_orders_dataset.csv,order_delivered_carrier_date,1783,1.7930,Rows,Expected by some statuses / Needs Investigation when delivered,Missingness varies by order_status; 2 delivered rows are missing this field.,None as a feature; lifecycle-quality context only,Retain for audit; investigate delivered-status contradictions
2,olist_orders_dataset.csv,order_delivered_customer_date,2965,2.9817,Rows,Expected by some statuses / Needs Investigation when delivered,Missingness varies by order_status; 8 delivered rows are missing this field.,Classification and regression target eligibility,Assess label eligibility; investigate delivered-status and canceled-with-delivery contradictions
3,olist_orders_dataset.csv,order_status vs order_delivered_customer_date,6,0.0060,Rows,Needs Investigation,"6 canceled orders have an actual customer-delivery timestamp (count of contradictions, not missi...",Open target-eligibility decision: timestamp-only vs status-based rule,Decide and document the eligibility rule in target construction; report the count
4,olist_products_dataset.csv,product_category_name,610,1.8512,Rows,Needs Investigation,Descriptive fields show coordinated absence; compare the all-four-missing pattern.,Potential predictive feature availability,Retain; document missing indicator and consider later train-only treatment
5,olist_products_dataset.csv,product_name_lenght,610,1.8512,Rows,Needs Investigation,Descriptive fields show coordinated absence; compare the all-four-missing pattern.,Potential predictive feature availability,Retain; document missing indicator and consider later train-only treatment
6,olist_products_dataset.csv,product_description_lenght,610,1.8512,Rows,Needs Investigation,Descriptive fields show coordinated absence; compare the all-four-missing pattern.,Potential predictive feature availability,Retain; document missing indicator and consider later train-only treatment
7,olist_products_dataset.csv,product_photos_qty,610,1.8512,Rows,Needs Investigation,Descriptive fields show coordinated absence; compare the all-four-missing pattern.,Potential predictive feature availability,Retain; document missing indicator and consider later train-only treatment
8,olist_products_dataset.csv,product_weight_g,2,0.0061,Rows,Needs Investigation,All four physical measurements are absent together for a very small product set.,Potential predictive feature availability,Investigate source records; consider later train-only treatment
9,olist_products_dataset.csv,product_length_cm,2,0.0061,Rows,Needs Investigation,All four physical measurements are absent together for a very small product set.,Potential predictive feature availability,Investigate source records; consider later train-only treatment


### Recommendations

1. Keep lifecycle missingness conditional on `order_status`; do not label every undelivered order as defective.
2. Investigate delivered-status orders missing actual delivery before target construction, and report all target-eligibility exclusions.
3. Record an explicit eligibility rule for canceled orders that carry an actual customer-delivery timestamp (timestamp-only vs status-based) during target construction, with the affected count.
4. Preserve the coordinated product-metadata pattern as evidence for later feature-treatment design; do not fill it in this task.
5. Investigate the isolated products without physical measurements before deciding on training-only preprocessing.
6. Treat review text as optional retrospective data and exclude the entire review source from purchase-time features.
7. Preserve distinctions among `NaN`, blank strings, whitespace, and suspected placeholder tokens; interpret literal categories in business context. Check hidden missingness against raw file text, because the default CSV loader converts common NA tokens before any string check can see them.

## 11. Learning Questions

These questions are intentionally unanswered.

1. Why is a missing value not automatically a data-quality problem?
2. Why should missing delivery dates be investigated against order status?
3. What is the difference between expected and suspicious missingness?
4. Why should we not blindly call `dropna()`?
5. How can missing target values affect training eligibility?
6. How is a missing model feature different from a missing target?
7. Why should review information not be used for purchase-time prediction?
8. What can patterns of missing values reveal about data collection?

## 12. Duplicate Validation

### 12.1 Objectives and Key Concepts

This section implements **Task 2 only: duplicate validation** across all nine source datasets. It does not remove rows or build the final modeling table.

**What are we checking?** Four related but distinct ideas:

- An **exact duplicate row** repeats every column value.
- A **duplicate identifier** repeats a value in a column proposed as a key.
- A **duplicate composite key** repeats the same combination of two or more key columns.
- A **legitimate one-to-many relationship** repeats a parent identifier because several child records belong to one parent.

**Why does it matter?** Duplicate interpretation depends on dataset grain. A repeated `order_id` would contradict the Orders grain, but is normal in Order Items and Payments. Confusing valid child rows with duplicate errors can destroy information; ignoring an unexpected key duplicate can multiply training observations.

**What should we understand?** `duplicated()` detects repeated observations according to the columns supplied. It does not decide whether the repetition is valid—the grain and business relationship provide that meaning.

### 12.2 Dataset Grain and Expected Unique Keys

**What are we checking?** The intended meaning of one row, the key or candidate key appropriate to that grain, and identifiers that may repeat legitimately.

**Why does it matter?** A uniqueness test is meaningful only after the expected grain is stated.

**What should we understand?** “Observed candidate key” means uniqueness is supported by this snapshot but is not automatically a permanent source-system guarantee. Reviews and Geolocation require special treatment because their individual identifiers or ZIP prefixes are not unique by design in the observed data.

In [13]:
DUPLICATE_GRAIN_RULES = {
    "Orders": {
        "dataframe": orders,
        "row_grain": "One order lifecycle record",
        "expected_key": ["order_id"],
        "key_status": "Expected unique primary key",
        "legitimate_repetition": "Status and customer-independent attributes may repeat; order_id should not",
    },
    "Order Items": {
        "dataframe": order_items,
        "row_grain": "One item sequence within an order",
        "expected_key": ["order_id", "order_item_id"],
        "key_status": "Observed candidate composite key",
        "legitimate_repetition": "order_id repeats for multi-item orders",
    },
    "Customers": {
        "dataframe": customers,
        "row_grain": "One order-specific customer record",
        "expected_key": ["customer_id"],
        "key_status": "Expected unique primary key",
        "legitimate_repetition": "customer_unique_id may repeat across orders",
    },
    "Products": {
        "dataframe": products,
        "row_grain": "One catalog product",
        "expected_key": ["product_id"],
        "key_status": "Expected unique primary key",
        "legitimate_repetition": "Category and physical attributes may repeat across products",
    },
    "Sellers": {
        "dataframe": sellers,
        "row_grain": "One marketplace seller",
        "expected_key": ["seller_id"],
        "key_status": "Expected unique primary key",
        "legitimate_repetition": "Location values may repeat across sellers",
    },
    "Payments": {
        "dataframe": payments,
        "row_grain": "One payment sequence within an order",
        "expected_key": ["order_id", "payment_sequential"],
        "key_status": "Observed candidate composite key",
        "legitimate_repetition": "order_id repeats for orders with multiple payment rows",
    },
    "Reviews": {
        "dataframe": reviews,
        "row_grain": "One review record; individual IDs are not assumed unique",
        "expected_key": ["review_id", "order_id"],
        "key_status": "Observed candidate pair; validate review_id and order_id separately",
        "legitimate_repetition": "An order may have multiple review rows; review_id also repeats in this snapshot",
    },
    "Geolocation": {
        "dataframe": geolocation,
        "row_grain": "One raw coordinate/locality observation",
        "expected_key": ["geolocation_zip_code_prefix"],
        "key_status": "Not a unique key; repetition is expected",
        "legitimate_repetition": "ZIP prefixes and coordinate/locality observations may repeat",
    },
    "Category Translation": {
        "dataframe": category_translation,
        "row_grain": "One Portuguese-to-English category mapping",
        "expected_key": ["product_category_name"],
        "key_status": "Expected unique lookup key",
        "legitimate_repetition": "No key repetition expected",
    },
}

grain_summary = pd.DataFrame(
    [
        {
            "Dataset": dataset_name,
            "Row Grain": rule["row_grain"],
            "Expected Key": " + ".join(rule["expected_key"]),
            "Key Status": rule["key_status"],
            "Legitimate Repetition": rule["legitimate_repetition"],
        }
        for dataset_name, rule in DUPLICATE_GRAIN_RULES.items()
    ]
)
display(grain_summary)

,Dataset,Row Grain,Expected Key,Key Status,Legitimate Repetition
0,Orders,One order lifecycle record,order_id,Expected unique primary key,Status and customer-independent attributes may repeat; order_id should not
1,Order Items,One item sequence within an order,order_id + order_item_id,Observed candidate composite key,order_id repeats for multi-item orders
2,Customers,One order-specific customer record,customer_id,Expected unique primary key,customer_unique_id may repeat across orders
3,Products,One catalog product,product_id,Expected unique primary key,Category and physical attributes may repeat across products
4,Sellers,One marketplace seller,seller_id,Expected unique primary key,Location values may repeat across sellers
5,Payments,One payment sequence within an order,order_id + payment_sequential,Observed candidate composite key,order_id repeats for orders with multiple payment rows
6,Reviews,One review record; individual IDs are not assumed unique,review_id + order_id,Observed candidate pair; validate review_id and order_id separately,An order may have multiple review rows; review_id also repeats in this snapshot
7,Geolocation,One raw coordinate/locality observation,geolocation_zip_code_prefix,Not a unique key; repetition is expected,ZIP prefixes and coordinate/locality observations may repeat
8,Category Translation,One Portuguese-to-English category mapping,product_category_name,Expected unique lookup key,No key repetition expected


### 12.3 Exact Duplicate Rows — All Nine Datasets

**What are we checking?** Rows whose values match a previous row across every column, using `dataframe.duplicated().sum()`.

**Why does it matter?** Exact duplicate rows can unintentionally overweight observations, but they can also reflect a source's sampling structure. They are different from repeated identifiers: two rows may share an ID while differing in other fields.

**What should we understand?** The count uses `keep="first"` behavior, so it reports repeated occurrences beyond the first copy. The percentage uses total dataset rows as its denominator. No rows are removed.

In [14]:
def exact_duplicate_profile(
    dataset_name: str, dataframe: pd.DataFrame
) -> dict:
    '''Summarize repeated full rows without changing the DataFrame.'''
    total_rows = len(dataframe)
    duplicate_rows = int(dataframe.duplicated().sum())
    return {
        "Dataset": dataset_name,
        "Total Rows": total_rows,
        "Exact Duplicates": duplicate_rows,
        "Exact Duplicate Percentage": (
            duplicate_rows / total_rows * 100 if total_rows else 0.0
        ),
    }


exact_duplicate_summary = pd.DataFrame(
    [
        exact_duplicate_profile(dataset_name, dataframe)
        for dataset_name, dataframe in datasets.items()
    ]
)
display(exact_duplicate_summary)

,Dataset,Total Rows,Exact Duplicates,Exact Duplicate Percentage
0,Orders,99441,0,0.0000
1,Order Items,112650,0,0.0000
2,Products,32951,0,0.0000
3,Customers,99441,0,0.0000
4,Sellers,3095,0,0.0000
5,Payments,103886,0,0.0000
6,Reviews,99224,0,0.0000
7,Geolocation,1000163,261831,26.1788
8,Category Translation,71,0,0.0000


**Interpretation guidance:** A zero means no complete row repeats in that source snapshot. Geolocation requires context: repeated complete observations may arise from its raw geographic sampling structure, so this result supports careful aggregation design—not automatic deletion.

### 12.4 Primary and Composite Key Validation

**What are we checking?** Unique key combinations, duplicate occurrences beyond the first, repeated key groups, affected records, and whether each tested uniqueness rule holds.

**Why does it matter?** A single-column key and a composite key answer different questions. `order_id` legitimately repeats in child tables, while (`order_id`, sequence number) should identify a specific item or payment record in this snapshot.

**What should we understand?** Reviews are tested three ways—`review_id`, `order_id`, and their pair—so observed cardinality is documented rather than assumed. Geolocation ZIP repetition is measured but is not treated as a key violation because ZIP is not the raw table's unique grain.

In [15]:
KEY_CHECKS = [
    ("Orders", ["order_id"], "Expected unique"),
    ("Order Items", ["order_id"], "Expected to repeat"),
    ("Order Items", ["order_id", "order_item_id"], "Expected unique composite"),
    ("Customers", ["customer_id"], "Expected unique"),
    ("Customers", ["customer_unique_id"], "Expected to repeat across orders"),
    ("Products", ["product_id"], "Expected unique"),
    ("Sellers", ["seller_id"], "Expected unique"),
    ("Payments", ["order_id"], "Expected to repeat"),
    ("Payments", ["order_id", "payment_sequential"], "Expected unique composite"),
    ("Reviews", ["review_id"], "Cardinality to investigate"),
    ("Reviews", ["order_id"], "Cardinality to investigate"),
    ("Reviews", ["review_id", "order_id"], "Observed candidate composite"),
    ("Geolocation", ["geolocation_zip_code_prefix"], "Expected to repeat"),
    ("Category Translation", ["product_category_name"], "Expected unique"),
]


def key_duplicate_profile(
    dataset_name: str,
    dataframe: pd.DataFrame,
    key_columns: list[str],
    expectation: str,
) -> dict:
    '''Measure repetition for a candidate single or composite key.

    Group sizes preserve missing-key groups for completeness. Missing-key meaning
    was investigated in Task 1 and is not reclassified here.
    '''
    key_group_sizes = dataframe.groupby(
        key_columns, dropna=False, sort=False
    ).size()
    repeated_groups = key_group_sizes[key_group_sizes > 1]
    duplicate_occurrences = int((key_group_sizes - 1).clip(lower=0).sum())
    return {
        "Dataset": dataset_name,
        "Key": " + ".join(key_columns),
        "Expectation": expectation,
        "Total Rows": len(dataframe),
        "Unique Key Combinations": int(len(key_group_sizes)),
        "Duplicate Occurrences": duplicate_occurrences,
        "Repeated Key Groups": int(len(repeated_groups)),
        "Affected Records": int(repeated_groups.sum()),
        "Uniqueness Holds": duplicate_occurrences == 0,
    }


key_validation = pd.DataFrame(
    [
        key_duplicate_profile(
            dataset_name,
            datasets[dataset_name],
            key_columns,
            expectation,
        )
        for dataset_name, key_columns, expectation in KEY_CHECKS
    ]
)
display(key_validation)

,Dataset,Key,Expectation,Total Rows,Unique Key Combinations,Duplicate Occurrences,Repeated Key Groups,Affected Records,Uniqueness Holds
0,Orders,order_id,Expected unique,99441,99441,0,0,0,True
1,Order Items,order_id,Expected to repeat,112650,98666,13984,9803,23787,False
2,Order Items,order_id + order_item_id,Expected unique composite,112650,112650,0,0,0,True
3,Customers,customer_id,Expected unique,99441,99441,0,0,0,True
4,Customers,customer_unique_id,Expected to repeat across orders,99441,96096,3345,2997,6342,False
5,Products,product_id,Expected unique,32951,32951,0,0,0,True
6,Sellers,seller_id,Expected unique,3095,3095,0,0,0,True
7,Payments,order_id,Expected to repeat,103886,99440,4446,2961,7407,False
8,Payments,order_id + payment_sequential,Expected unique composite,103886,103886,0,0,0,True
9,Reviews,review_id,Cardinality to investigate,99224,98410,814,789,1603,False


In [16]:
# Samples are limited to repeated identifiers that need interpretation. Tables
# whose expected primary/composite keys are unique have no suspicious sample.
repeated_review_id_mask = reviews["review_id"].duplicated(keep=False)
repeated_review_order_mask = reviews["order_id"].duplicated(keep=False)
exact_geolocation_duplicate_mask = geolocation.duplicated(keep=False)

display(
    reviews.loc[repeated_review_id_mask]
    .sort_values(["review_id", "order_id"])
    .head(10)
)
display(
    reviews.loc[repeated_review_order_mask]
    .sort_values(["order_id", "review_id"])
    .head(10)
)
display(
    geolocation.loc[exact_geolocation_duplicate_mask]
    .sort_values(list(geolocation.columns))
    .head(10)
)

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
29841,00130cbe1f9d422698c812ed8ded1919,04a28263e085d399c97ae49e0b477efa,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecido pela impressora, consequentemente não funcionou....",2018-03-07 00:00:00,2018-03-20 18:08:23
46678,00130cbe1f9d422698c812ed8ded1919,dfcdfc43867d1c1381bfaf62d6b9c195,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecido pela impressora, consequentemente não funcionou....",2018-03-07 00:00:00,2018-03-20 18:08:23
63193,0115633a9c298b6a98bcbe4eee75345f,0c9850b2c179c1ef60d2855e2751d1fa,5,NaN,NaN,2017-09-21 00:00:00,2017-09-26 03:27:47
90677,0115633a9c298b6a98bcbe4eee75345f,78a4201f58af3463bdab842eea4bc801,5,NaN,NaN,2017-09-21 00:00:00,2017-09-26 03:27:47
57280,0174caf0ee5964646040cd94e15ac95e,74db91e33b4e1fd865356c89a61abf1f,1,NaN,Produto entregue dentro de embalagem do fornecedor sem os parafusos de fixação das partes.,2018-03-07 00:00:00,2018-03-08 03:00:53
92876,0174caf0ee5964646040cd94e15ac95e,f93a732712407c02dce5dd5088d0f47b,1,NaN,Produto entregue dentro de embalagem do fornecedor sem os parafusos de fixação das partes.,2018-03-07 00:00:00,2018-03-08 03:00:53
54832,017808d29fd1f942d97e50184dfb4c13,8daaa9e99d60fbba579cc1c3e3bfae01,5,NaN,NaN,2018-03-02 00:00:00,2018-03-05 01:43:30
99167,017808d29fd1f942d97e50184dfb4c13,b1461c8882153b5fe68307c46a506e39,5,NaN,NaN,2018-03-02 00:00:00,2018-03-05 01:43:30
96080,0254bd905dc677a6078990aad3331a36,331b367bdd766f3d1cf518777317b5d9,1,NaN,"O pedido consta de 2 produtos e até agora recebi apenas 1 produto, e o que me preocupa é que o s...",2017-09-09 00:00:00,2017-09-13 09:52:44
20621,0254bd905dc677a6078990aad3331a36,5bf226cf882c5bf4247f89a97c86f273,1,NaN,"O pedido consta de 2 produtos e até agora recebi apenas 1 produto, e o que me preocupa é que o s...",2017-09-09 00:00:00,2017-09-13 09:52:44


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
22423,2a74b0559eb58fc1ff842ecc999594cb,0035246a40f520710769010f752e7507,5,NaN,"Estou acostumada a comprar produtos pelo baratheon. Super confiável, transparência, qualidade e ...",2017-08-25 00:00:00,2017-08-29 21:45:57
25612,89a02c45c340aeeb1354a24e7d4b2c1e,0035246a40f520710769010f752e7507,5,NaN,NaN,2017-08-29 00:00:00,2017-08-30 01:59:12
68633,73413b847f63e02bc752b364f6d05ee9,013056cfe49763c6f66bda03396c5ee3,4,NaN,NaN,2018-03-04 00:00:00,2018-03-05 17:02:00
22779,ab30810c29da5da8045216f0f62652a2,013056cfe49763c6f66bda03396c5ee3,5,NaN,NaN,2018-02-22 00:00:00,2018-02-23 12:12:30
854,830636803620cdf8b6ffaf1b2f6e92b2,0176a6846bcb3b0d3aa3116a9a768597,5,NaN,NaN,2017-12-30 00:00:00,2018-01-02 10:54:06
83224,d8e8c42271c8fb67b9dad95d98c8ff80,0176a6846bcb3b0d3aa3116a9a768597,5,NaN,NaN,2017-12-30 00:00:00,2018-01-02 10:54:47
17582,017f0e1ea6386de662cbeba299c59ad1,02355020fd0a40a0d56df9f6ff060413,1,NaN,ja reclamei varias vezes e ate hoje não sei onde foi parar o produto que comprei. quero cancelar...,2018-03-29 00:00:00,2018-03-30 03:16:19
89888,0c8e7347f1cdd2aede37371543e3d163,02355020fd0a40a0d56df9f6ff060413,3,NaN,UM DOS PRODUTOS (ENTREGA02) COMPRADOS NESTE PEDIDO NAO RECEBI AINDA E OUTRO (ENTREGA01) RECEBI E...,2018-03-21 00:00:00,2018-03-22 01:32:08
37911,04d945e95c788a3aa1ffbee42105637b,029863af4b968de1e5d6a82782e662f5,5,NaN,NaN,2017-07-14 00:00:00,2017-07-17 13:58:06
55137,61fe4e7d1ae801bbe169eb67b86c6eda,029863af4b968de1e5d6a82782e662f5,4,NaN,NaN,2017-07-19 00:00:00,2017-07-20 12:06:11


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
519,1001,-23.5513,-46.6340,sao paulo,SP
583,1001,-23.5513,-46.6340,sao paulo,SP
818,1001,-23.5513,-46.6340,sao paulo,SP
206,1001,-23.5505,-46.6343,sao paulo,SP
429,1001,-23.5505,-46.6343,sao paulo,SP
596,1001,-23.5505,-46.6343,sao paulo,SP
639,1001,-23.5505,-46.6343,sao paulo,SP
771,1001,-23.5505,-46.6343,sao paulo,SP
912,1001,-23.5505,-46.6343,sao paulo,SP
985,1001,-23.5505,-46.6343,sao paulo,SP


**Interpretation guidance:** Repeated `review_id` and `order_id` values show that neither field alone is a safe review-table primary key. A unique observed pair is evidence about this snapshot, not proof of a source-system guarantee. Exact geolocation repetition is visible evidence, but the raw observations must be understood and converted to a documented lookup grain later rather than blindly deleted here.

### 12.5 Legitimate vs. Suspicious Repetition

**What are we checking?** Repetition in the context of the business grain: multi-item orders, multi-payment orders, multiple reviews, repeat customers, and repeated geographic observations.

**Why does it matter?** The same numerical pattern can be valid in one table and a key violation in another.

**What should we understand?** “Expected repetition” retains meaningful child records. “Suspicious duplication” denotes a violated expected key. “Needs investigation” is used where the source grain is non-unique or incompletely governed, as in Reviews and Geolocation.

In [17]:
def repeated_parent_summary(
    dataset_name: str, dataframe: pd.DataFrame, parent_column: str
) -> dict:
    '''Summarize valid parent-key repetition in a child or entity-history table.'''
    records_per_parent = dataframe.groupby(parent_column, dropna=False).size()
    return {
        "Dataset": dataset_name,
        "Repeated Identifier": parent_column,
        "Distinct Values": int(len(records_per_parent)),
        "Groups With Multiple Rows": int((records_per_parent > 1).sum()),
        "Maximum Rows Per Value": int(records_per_parent.max()),
    }


legitimate_repetition_summary = pd.DataFrame(
    [
        repeated_parent_summary("Order Items", order_items, "order_id"),
        repeated_parent_summary("Payments", payments, "order_id"),
        repeated_parent_summary("Reviews", reviews, "order_id"),
        repeated_parent_summary("Customers", customers, "customer_unique_id"),
        repeated_parent_summary(
            "Geolocation", geolocation, "geolocation_zip_code_prefix"
        ),
    ]
)
display(legitimate_repetition_summary)

repetition_classification = pd.DataFrame(
    [
        {
            "Pattern": "Orders.order_id repetition",
            "Classification": "Suspicious duplication if present",
            "Evidence": "Orders has one lifecycle record per order.",
        },
        {
            "Pattern": "Order Items.order_id repetition",
            "Classification": "Expected repetition",
            "Evidence": "One order may contain several item-sequence rows.",
        },
        {
            "Pattern": "Payments.order_id repetition",
            "Classification": "Expected repetition",
            "Evidence": "One order may contain several payment-sequence rows.",
        },
        {
            "Pattern": "Reviews.review_id or order_id repetition",
            "Classification": "Needs investigation",
            "Evidence": "Neither identifier is individually unique in the observed source.",
        },
        {
            "Pattern": "Geolocation ZIP and exact-row repetition",
            "Classification": "Expected repetition / Needs investigation before joins",
            "Evidence": "The file contains raw geographic observations, not one row per ZIP.",
        },
    ]
)
display(repetition_classification)

,Dataset,Repeated Identifier,Distinct Values,Groups With Multiple Rows,Maximum Rows Per Value
0,Order Items,order_id,98666,9803,21
1,Payments,order_id,99440,2961,29
2,Reviews,order_id,98673,547,3
3,Customers,customer_unique_id,96096,2997,17
4,Geolocation,geolocation_zip_code_prefix,19015,17972,1146


,Pattern,Classification,Evidence
0,Orders.order_id repetition,Suspicious duplication if present,Orders has one lifecycle record per order.
1,Order Items.order_id repetition,Expected repetition,One order may contain several item-sequence rows.
2,Payments.order_id repetition,Expected repetition,One order may contain several payment-sequence rows.
3,Reviews.review_id or order_id repetition,Needs investigation,Neither identifier is individually unique in the observed source.
4,Geolocation ZIP and exact-row repetition,Expected repetition / Needs investigation before joins,"The file contains raw geographic observations, not one row per ZIP."


### 12.6 Duplicate Implications for Dataset Joins

**What are we checking?** Native rows per `order_id` for the one-to-many sources, without joining them.

**Why does it matter?** Joining Orders directly to Items creates one row per item, not one row per order. Joining Payments or Reviews afterward can multiply those rows again—for example, an order with two items and three payment rows can produce six joined rows.

**What should we understand?** Items and Payments must be aggregated independently to order grain before the modeling join. Reviews require separate handling and are excluded from purchase-time features. Geolocation must become a documented one-row-per-ZIP lookup before customer or seller enrichment. This section diagnoses cardinality only; it does not build that dataset.

In [18]:
ORDER_CHILD_SOURCES = {
    "Order Items": order_items,
    "Payments": payments,
    "Reviews": reviews,
}

join_cardinality_rows = []
for dataset_name, dataframe in ORDER_CHILD_SOURCES.items():
    rows_per_order = dataframe.groupby("order_id", dropna=False).size()
    join_cardinality_rows.append(
        {
            "Dataset": dataset_name,
            "Source Rows": len(dataframe),
            "Distinct Orders": int(len(rows_per_order)),
            "Orders With Multiple Rows": int((rows_per_order > 1).sum()),
            "Maximum Rows Per Order": int(rows_per_order.max()),
            "Required Later Handling": (
                "Aggregate independently to one row per order"
                if dataset_name in {"Order Items", "Payments"}
                else "Keep retrospective and outside purchase-time features"
            ),
        }
    )

join_cardinality_summary = pd.DataFrame(join_cardinality_rows)
display(join_cardinality_summary)

geolocation_join_risk = pd.DataFrame(
    [
        {
            "Source": "Geolocation",
            "Rows": len(geolocation),
            "Distinct ZIP Prefixes": geolocation[
                "geolocation_zip_code_prefix"
            ].nunique(),
            "ZIP Prefixes With Multiple Rows": int(
                (
                    geolocation.groupby("geolocation_zip_code_prefix").size()
                    > 1
                ).sum()
            ),
            "Required Later Handling": "Define and validate one canonical row per ZIP before joining",
        }
    ]
)
display(geolocation_join_risk)

,Dataset,Source Rows,Distinct Orders,Orders With Multiple Rows,Maximum Rows Per Order,Required Later Handling
0,Order Items,112650,98666,9803,21,Aggregate independently to one row per order
1,Payments,103886,99440,2961,29,Aggregate independently to one row per order
2,Reviews,99224,98673,547,3,Keep retrospective and outside purchase-time features


,Source,Rows,Distinct ZIP Prefixes,ZIP Prefixes With Multiple Rows,Required Later Handling
0,Geolocation,1000163,19015,17972,Define and validate one canonical row per ZIP before joining


### 12.7 ML Impact Assessment

Duplicate behavior affects both targets even when target definitions are correct:

| Risk | Classification consequence | Regression consequence | Control required later |
|---|---|---|---|
| Unexpected duplicate order rows | Repeats the same `is_late` outcome and distorts late-delivery rates | Repeats the same duration and changes the empirical target distribution | Assert one final row per `order_id` |
| Raw one-to-many joins | Overweights orders with more items, payments, or review rows | Biases loss and error metrics toward repeated orders | Aggregate each child table independently before joining |
| Raw Geolocation join | Multiplies customer/seller rows by ZIP observations | Repeats target rows and corrupts route-level summaries | Build a validated unique ZIP lookup first |
| Same order across data splits | Leaks order-specific information and produces optimistic classification metrics | Produces optimistic duration errors | Split only after enforcing order grain; keep every `order_id` in one chronological partition |
| Legitimate child repetition removed as “duplicates” | Loses order-composition evidence | Loses price, freight, product, or payment information | Preserve child records and aggregate with explicit rules later |

Duplicate records and valid one-to-many rows are not interchangeable. The release-blocking condition is an unintended loss of the final one-row-per-order grain, not the mere presence of repeated parent identifiers in normalized source tables. No train/test split is implemented in this task.

### 12.8 Findings and Recommendations

**What are we checking?** A consolidated, calculated summary for every dataset.

**Why does it matter?** The final classification must combine exact-row evidence, key behavior, and business grain instead of relying on one duplicate count.

**What should we understand?** Recommended actions preserve source evidence. “Aggregate later” describes future join preparation and is not performed here.

In [19]:
key_validation_indexed = key_validation.set_index(["Dataset", "Key"])
exact_duplicate_indexed = exact_duplicate_summary.set_index("Dataset")

selected_key_evidence = {
    "Orders": ("order_id", "Expected", "Retain; assert uniqueness before order-level joins"),
    "Order Items": (
        "order_id + order_item_id",
        "Expected repetition / composite key confirmed",
        "Retain all item rows; aggregate later by order_id",
    ),
    "Customers": ("customer_id", "Expected", "Retain; join Orders through customer_id"),
    "Products": ("product_id", "Expected", "Retain; assert uniqueness before enrichment"),
    "Sellers": ("seller_id", "Expected", "Retain; assert uniqueness before enrichment"),
    "Payments": (
        "order_id + payment_sequential",
        "Expected repetition / composite key confirmed",
        "Retain all payment rows; aggregate later by order_id",
    ),
    "Reviews": (
        "review_id + order_id",
        "Needs Investigation",
        "Retain for retrospective analysis; do not assume either individual ID is unique",
    ),
    "Geolocation": (
        "geolocation_zip_code_prefix",
        "Expected repetition / Needs Investigation before joins",
        "Retain raw evidence; design a validated ZIP-level aggregation later",
    ),
    "Category Translation": (
        "product_category_name",
        "Expected",
        "Retain; assert lookup-key uniqueness before enrichment",
    ),
}

duplicate_findings_rows = []
for dataset_name, rule in DUPLICATE_GRAIN_RULES.items():
    selected_key, classification, recommended_action = selected_key_evidence[
        dataset_name
    ]
    key_result = key_validation_indexed.loc[(dataset_name, selected_key)]
    exact_duplicates = int(
        exact_duplicate_indexed.loc[dataset_name, "Exact Duplicates"]
    )

    if dataset_name == "Reviews":
        review_id_duplicates = int(
            key_validation_indexed.loc[
                ("Reviews", "review_id"), "Duplicate Occurrences"
            ]
        )
        order_id_duplicates = int(
            key_validation_indexed.loc[
                ("Reviews", "order_id"), "Duplicate Occurrences"
            ]
        )
        evidence = (
            f"review_id has {review_id_duplicates} repeated occurrences; "
            f"order_id has {order_id_duplicates}; the observed pair has "
            f"{int(key_result['Duplicate Occurrences'])}."
        )
    elif dataset_name == "Geolocation":
        evidence = (
            f"ZIP has {int(key_result['Duplicate Occurrences'])} repeated occurrences; "
            f"{exact_duplicates} full-row occurrences repeat beyond the first."
        )
    else:
        evidence = (
            f"Tested key has {int(key_result['Duplicate Occurrences'])} repeated "
            f"occurrences; full-row duplicates: {exact_duplicates}."
        )

    duplicate_findings_rows.append(
        {
            "Dataset": DATASET_FILES[dataset_name],
            "Row Grain": rule["row_grain"],
            "Expected Key": selected_key,
            "Exact Duplicates": exact_duplicates,
            "Key Duplicates": int(key_result["Duplicate Occurrences"]),
            "Classification": classification,
            "Evidence": evidence,
            "ML Impact": (
                "Join multiplication or repeated target rows if native grain is ignored"
                if dataset_name in {"Order Items", "Payments", "Reviews", "Geolocation"}
                else "Unexpected key duplicates would violate the intended enrichment/model grain"
            ),
            "Recommended Action": recommended_action,
        }
    )

duplicate_findings = pd.DataFrame(duplicate_findings_rows)
display(duplicate_findings)

,Dataset,Row Grain,Expected Key,Exact Duplicates,Key Duplicates,Classification,Evidence,ML Impact,Recommended Action
0,olist_orders_dataset.csv,One order lifecycle record,order_id,0,0,Expected,Tested key has 0 repeated occurrences; full-row duplicates: 0.,Unexpected key duplicates would violate the intended enrichment/model grain,Retain; assert uniqueness before order-level joins
1,olist_order_items_dataset.csv,One item sequence within an order,order_id + order_item_id,0,0,Expected repetition / composite key confirmed,Tested key has 0 repeated occurrences; full-row duplicates: 0.,Join multiplication or repeated target rows if native grain is ignored,Retain all item rows; aggregate later by order_id
2,olist_customers_dataset.csv,One order-specific customer record,customer_id,0,0,Expected,Tested key has 0 repeated occurrences; full-row duplicates: 0.,Unexpected key duplicates would violate the intended enrichment/model grain,Retain; join Orders through customer_id
3,olist_products_dataset.csv,One catalog product,product_id,0,0,Expected,Tested key has 0 repeated occurrences; full-row duplicates: 0.,Unexpected key duplicates would violate the intended enrichment/model grain,Retain; assert uniqueness before enrichment
4,olist_sellers_dataset.csv,One marketplace seller,seller_id,0,0,Expected,Tested key has 0 repeated occurrences; full-row duplicates: 0.,Unexpected key duplicates would violate the intended enrichment/model grain,Retain; assert uniqueness before enrichment
5,olist_order_payments_dataset.csv,One payment sequence within an order,order_id + payment_sequential,0,0,Expected repetition / composite key confirmed,Tested key has 0 repeated occurrences; full-row duplicates: 0.,Join multiplication or repeated target rows if native grain is ignored,Retain all payment rows; aggregate later by order_id
6,olist_order_reviews_dataset.csv,One review record; individual IDs are not assumed unique,review_id + order_id,0,0,Needs Investigation,review_id has 814 repeated occurrences; order_id has 551; the observed pair has 0.,Join multiplication or repeated target rows if native grain is ignored,Retain for retrospective analysis; do not assume either individual ID is unique
7,olist_geolocation_dataset.csv,One raw coordinate/locality observation,geolocation_zip_code_prefix,261831,981148,Expected repetition / Needs Investigation before joins,ZIP has 981148 repeated occurrences; 261831 full-row occurrences repeat beyond the first.,Join multiplication or repeated target rows if native grain is ignored,Retain raw evidence; design a validated ZIP-level aggregation later
8,product_category_name_translation.csv,One Portuguese-to-English category mapping,product_category_name,0,0,Expected,Tested key has 0 repeated occurrences; full-row duplicates: 0.,Unexpected key duplicates would violate the intended enrichment/model grain,Retain; assert lookup-key uniqueness before enrichment


### 12.9 Learning Questions

These questions are intentionally unanswered.

1. What is the difference between duplicated rows and duplicated IDs?
2. Why is `order_id` expected to be unique in Orders but repeated in Order Items?
3. What is a composite key?
4. Why should we not automatically remove repeated geolocation records?
5. How can duplicate keys affect Pandas joins?
6. How can duplicated orders bias model training?
7. Why is understanding dataset grain necessary before checking duplicates?
8. What is the difference between duplicate data and legitimate one-to-many relationships?

## 13. Data Type Validation

### 13.1 Objectives and Key Concepts

This section implements **Task 3 only: data type validation** across all nine datasets. It compares Pandas' in-memory storage types with each column's business meaning and intended analytical representation. All conversions are temporary validation objects; no source DataFrame or CSV column is overwritten.

**Validate before converting.** The Task 1 setup cell already converted `order_approved_at`, `order_delivered_carrier_date`, and `order_delivered_customer_date` with `errors="coerce"`. Any unparseable value in those columns would already be `NaT`, indistinguishable from a genuinely missing value. This section therefore reloads all nine CSVs unmodified (`raw_datasets`) so that every type check sees the values exactly as Pandas infers them from the files.

Three type concepts must remain separate:

1. **Storage type:** how Pandas currently stores the values, such as `str`, `int64`, `float64`, or `datetime64[us]`.
2. **Semantic type:** what the values mean, such as an identifier, timestamp, monetary measurement, category, text, or geographic code.
3. **ML representation:** how an approved purchase-time feature might eventually be encoded for a model. That later representation is not implemented here.

A numeric-looking value is not necessarily a continuous feature. ZIP prefixes and sequence identifiers are examples: arithmetic distance between their integer values has no useful business interpretation.

### 13.2 Current Data Types — All Nine Datasets

**What are we checking?** The current Pandas dtype, non-null count, and null count for every source column, as inferred from an unmodified CSV load.

**Why are we checking it?** CSV files do not carry a complete analytical schema. Pandas infers storage types during loading, and inference can differ from business meaning.

**What should we understand?** A `str` or `object` dtype is appropriate for many identifiers, categories, and text fields. It becomes a concern only when the intended operation—such as datetime arithmetic—requires a validated conversion.

In [20]:
# The setup cell converted three Orders timestamp columns in place for Task 1.
# Validating those converted columns would hide failures: an unparseable string
# would already be NaT and be miscounted as an original null. Type validation
# therefore uses a fresh, unmodified load of every source CSV.
raw_datasets = {
    dataset_name: load_source_csv(file_name)
    for dataset_name, file_name in DATASET_FILES.items()
}


def current_dtype_profile(
    dataset_name: str, dataframe: pd.DataFrame
) -> pd.DataFrame:
    '''Return concise dtype and completeness evidence for every column.'''
    return pd.DataFrame(
        {
            "Dataset": dataset_name,
            "Column": dataframe.columns,
            "Pandas Type": [str(dtype) for dtype in dataframe.dtypes],
            "Non-Null Count": dataframe.notna().sum().to_numpy(),
            "Null Count": dataframe.isna().sum().to_numpy(),
        }
    )


current_dtype_summary = pd.concat(
    [
        current_dtype_profile(dataset_name, dataframe)
        for dataset_name, dataframe in raw_datasets.items()
    ],
    ignore_index=True,
)
display(current_dtype_summary)

,Dataset,Column,Pandas Type,Non-Null Count,Null Count
0,Orders,order_id,str,99441,0
1,Orders,customer_id,str,99441,0
2,Orders,order_status,str,99441,0
3,Orders,order_purchase_timestamp,str,99441,0
4,Orders,order_approved_at,str,99281,160
5,Orders,order_delivered_carrier_date,str,97658,1783
6,Orders,order_delivered_customer_date,str,96476,2965
7,Orders,order_estimated_delivery_date,str,99441,0
8,Order Items,order_id,str,112650,0
9,Order Items,order_item_id,int64,112650,0


### 13.3 Expected Data Types and Semantic Roles

**What are we checking?** A documented expected analytical type and semantic role for all columns, based on `DATA_DICTIONARY.md` and the source schema.

**Why are we checking it?** Compatibility cannot be judged from dtype names alone. For example, an integer is suitable storage for a count but risky storage for a ZIP code if leading zeros matter.

**What should we understand?** “Expected type” describes a suitable analytical representation, not a conversion performed by this notebook. The schema check also guards against silently omitting a source column from the assessment.

In [21]:
EXPECTED_COLUMN_TYPES = {
    "Orders": {
        "order_id": ("String", "Identifier"),
        "customer_id": ("String", "Identifier"),
        "order_status": ("Category", "Categorical"),
        "order_purchase_timestamp": ("Datetime", "Datetime"),
        "order_approved_at": ("Datetime", "Datetime"),
        "order_delivered_carrier_date": ("Datetime", "Datetime"),
        "order_delivered_customer_date": ("Datetime", "Datetime"),
        "order_estimated_delivery_date": ("Datetime", "Datetime"),
    },
    "Order Items": {
        "order_id": ("String", "Identifier"),
        "order_item_id": ("Integer sequence", "Identifier"),
        "product_id": ("String", "Identifier"),
        "seller_id": ("String", "Identifier"),
        "shipping_limit_date": ("Datetime", "Datetime"),
        "price": ("Decimal numeric", "Numeric measurement"),
        "freight_value": ("Decimal numeric", "Numeric measurement"),
    },
    "Products": {
        "product_id": ("String", "Identifier"),
        "product_category_name": ("Category", "Categorical"),
        "product_name_lenght": ("Integer count", "Count"),
        "product_description_lenght": ("Integer count", "Count"),
        "product_photos_qty": ("Integer count", "Count"),
        "product_weight_g": ("Decimal numeric", "Numeric measurement"),
        "product_length_cm": ("Decimal numeric", "Numeric measurement"),
        "product_height_cm": ("Decimal numeric", "Numeric measurement"),
        "product_width_cm": ("Decimal numeric", "Numeric measurement"),
    },
    "Customers": {
        "customer_id": ("String", "Identifier"),
        "customer_unique_id": ("String", "Identifier"),
        "customer_zip_code_prefix": ("String/code", "Geographic identifier"),
        "customer_city": ("Category", "Categorical"),
        "customer_state": ("Category", "Categorical"),
    },
    "Sellers": {
        "seller_id": ("String", "Identifier"),
        "seller_zip_code_prefix": ("String/code", "Geographic identifier"),
        "seller_city": ("Category", "Categorical"),
        "seller_state": ("Category", "Categorical"),
    },
    "Payments": {
        "order_id": ("String", "Identifier"),
        "payment_sequential": ("Integer sequence", "Identifier"),
        "payment_type": ("Category", "Categorical"),
        "payment_installments": ("Integer count", "Count"),
        "payment_value": ("Decimal numeric", "Numeric measurement"),
    },
    "Reviews": {
        "review_id": ("String", "Identifier"),
        "order_id": ("String", "Identifier"),
        "review_score": ("Integer ordinal", "Ordinal categorical"),
        "review_comment_title": ("String", "Text"),
        "review_comment_message": ("String", "Text"),
        "review_creation_date": ("Datetime", "Datetime"),
        "review_answer_timestamp": ("Datetime", "Datetime"),
    },
    "Geolocation": {
        "geolocation_zip_code_prefix": ("String/code", "Geographic identifier"),
        "geolocation_lat": ("Decimal numeric", "Numeric measurement"),
        "geolocation_lng": ("Decimal numeric", "Numeric measurement"),
        "geolocation_city": ("Category", "Categorical"),
        "geolocation_state": ("Category", "Categorical"),
    },
    "Category Translation": {
        "product_category_name": ("Category", "Categorical"),
        "product_category_name_english": ("Category", "Categorical"),
    },
}

expected_type_rows = []
for dataset_name, dataframe in raw_datasets.items():
    documented_columns = set(EXPECTED_COLUMN_TYPES[dataset_name])
    observed_columns = set(dataframe.columns)
    if documented_columns != observed_columns:
        raise ValueError(
            f"Type specification mismatch for {dataset_name}: "
            f"missing={observed_columns - documented_columns}, "
            f"unexpected={documented_columns - observed_columns}"
        )

    for column, (expected_type, semantic_role) in EXPECTED_COLUMN_TYPES[
        dataset_name
    ].items():
        expected_type_rows.append(
            {
                "Dataset": dataset_name,
                "Column": column,
                "Expected Type": expected_type,
                "Semantic Role": semantic_role,
            }
        )

expected_type_summary = pd.DataFrame(expected_type_rows)
display(expected_type_summary)

,Dataset,Column,Expected Type,Semantic Role
0,Orders,order_id,String,Identifier
1,Orders,customer_id,String,Identifier
2,Orders,order_status,Category,Categorical
3,Orders,order_purchase_timestamp,Datetime,Datetime
4,Orders,order_approved_at,Datetime,Datetime
5,Orders,order_delivered_carrier_date,Datetime,Datetime
6,Orders,order_delivered_customer_date,Datetime,Datetime
7,Orders,order_estimated_delivery_date,Datetime,Datetime
8,Order Items,order_id,String,Identifier
9,Order Items,order_item_id,Integer sequence,Identifier


### 13.4 Data Type Mismatch Investigation

**What are we checking?** Whether the current storage dtype is compatible with the documented semantic role.

**Why are we checking it?** Reporting every string column as a mismatch would be misleading. Identifiers and categories can be stored correctly as strings even though a future model will require a separate representation.

**What should we understand?** Results are classified as **Correct / Compatible**, **Conversion Recommended Later**, or **Suspicious / Needs Investigation**. Timestamp strings and numeric ZIP storage can be valid raw representations while still requiring controlled later conversion.

In [22]:
def classify_dtype_compatibility(dtype, semantic_role: str) -> str:
    '''Classify storage compatibility without converting the source column.'''
    is_numeric = pd.api.types.is_numeric_dtype(dtype)
    is_datetime = pd.api.types.is_datetime64_any_dtype(dtype)
    is_text_like = pd.api.types.is_string_dtype(dtype) or pd.api.types.is_object_dtype(dtype)

    if semantic_role == "Datetime":
        return "Correct / Compatible" if is_datetime else "Conversion Recommended Later"
    if semantic_role in {"Numeric measurement", "Count", "Ordinal categorical"}:
        return "Correct / Compatible" if is_numeric else "Suspicious / Needs Investigation"
    if semantic_role == "Geographic identifier":
        # Numeric storage is parseable but can erase leading-zero formatting.
        return "Conversion Recommended Later" if is_numeric else "Correct / Compatible"
    if semantic_role in {"Identifier", "Categorical", "Text"}:
        # Sequence identifiers are legitimately integer-backed; opaque IDs are strings.
        return "Correct / Compatible" if is_text_like or is_numeric else "Suspicious / Needs Investigation"
    return "Suspicious / Needs Investigation"


dtype_comparison = expected_type_summary.merge(
    current_dtype_summary,
    on=["Dataset", "Column"],
    validate="one_to_one",
)
dtype_comparison["Initial Validation Result"] = [
    classify_dtype_compatibility(
        raw_datasets[dataset_name][column].dtype,
        semantic_role,
    )
    for dataset_name, column, semantic_role in dtype_comparison[
        ["Dataset", "Column", "Semantic Role"]
    ].itertuples(index=False, name=None)
]
display(dtype_comparison)

,Dataset,Column,Expected Type,Semantic Role,Pandas Type,Non-Null Count,Null Count,Initial Validation Result
0,Orders,order_id,String,Identifier,str,99441,0,Correct / Compatible
1,Orders,customer_id,String,Identifier,str,99441,0,Correct / Compatible
2,Orders,order_status,Category,Categorical,str,99441,0,Correct / Compatible
3,Orders,order_purchase_timestamp,Datetime,Datetime,str,99441,0,Conversion Recommended Later
4,Orders,order_approved_at,Datetime,Datetime,str,99281,160,Conversion Recommended Later
5,Orders,order_delivered_carrier_date,Datetime,Datetime,str,97658,1783,Conversion Recommended Later
6,Orders,order_delivered_customer_date,Datetime,Datetime,str,96476,2965,Conversion Recommended Later
7,Orders,order_estimated_delivery_date,Datetime,Datetime,str,99441,0,Conversion Recommended Later
8,Order Items,order_id,String,Identifier,str,112650,0,Correct / Compatible
9,Order Items,order_item_id,Integer sequence,Identifier,int64,112650,0,Correct / Compatible


### 13.5 Datetime Conversion Validation

**What are we checking?** Whether every non-null timestamp value can be parsed with `pd.to_datetime(errors="coerce")`.

**Why are we checking it?** Classification eligibility depends on actual and estimated delivery timestamps, while regression duration depends on purchase and actual delivery timestamps. Failed parsing can invalidate target construction.

**What should we understand?** Original missing values are excluded from the failure count. `errors="coerce"` temporarily maps unparseable non-null text to `NaT`, making failures measurable without overwriting the source. This task validates parsing only; chronology belongs to Task 5.

In [23]:
DATETIME_COLUMNS = {
    "Orders": [
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
    ],
    "Order Items": ["shipping_limit_date"],
    "Reviews": ["review_creation_date", "review_answer_timestamp"],
}


def datetime_conversion_profile(
    dataset_name: str, dataframe: pd.DataFrame, column: str
) -> tuple[dict, pd.DataFrame]:
    '''Test datetime parsing while excluding original nulls from failures.'''
    original = dataframe[column]
    converted = pd.to_datetime(original, errors="coerce")
    original_non_null = original.notna()
    failure_mask = original_non_null & converted.isna()
    non_null_count = int(original_non_null.sum())
    successful_count = int((original_non_null & converted.notna()).sum())

    summary = {
        "Dataset": dataset_name,
        "Column": column,
        "Current Type": str(original.dtype),
        "Original Non-Null": non_null_count,
        "Successful Conversions": successful_count,
        "Conversion Failures": int(failure_mask.sum()),
        "Success Percentage": (
            successful_count / non_null_count * 100 if non_null_count else 100.0
        ),
    }
    failures = pd.DataFrame(
        {
            "Dataset": dataset_name,
            "Column": column,
            "Unparseable Value": original.loc[failure_mask]
            .astype("string")
            .unique()[:5],
        }
    )
    return summary, failures


datetime_summaries = []
datetime_failure_samples = []
for dataset_name, columns in DATETIME_COLUMNS.items():
    for column in columns:
        summary, failures = datetime_conversion_profile(
            dataset_name, raw_datasets[dataset_name], column
        )
        datetime_summaries.append(summary)
        datetime_failure_samples.append(failures)

datetime_conversion_summary = pd.DataFrame(datetime_summaries)
datetime_conversion_failures = pd.concat(
    datetime_failure_samples, ignore_index=True
)
display(datetime_conversion_summary)
display(datetime_conversion_failures)

,Dataset,Column,Current Type,Original Non-Null,Successful Conversions,Conversion Failures,Success Percentage
0,Orders,order_purchase_timestamp,str,99441,99441,0,100.0000
1,Orders,order_approved_at,str,99281,99281,0,100.0000
2,Orders,order_delivered_carrier_date,str,97658,97658,0,100.0000
3,Orders,order_delivered_customer_date,str,96476,96476,0,100.0000
4,Orders,order_estimated_delivery_date,str,99441,99441,0,100.0000
5,Order Items,shipping_limit_date,str,112650,112650,0,100.0000
6,Reviews,review_creation_date,str,99224,99224,0,100.0000
7,Reviews,review_answer_timestamp,str,99224,99224,0,100.0000


,Dataset,Column,Unparseable Value


**Interpretation guidance:** A zero failure count means all present values are parseable under Pandas' parser; it does not prove correct event ordering, timezone semantics, or business validity. Empty failure samples are expected when parsing succeeds completely.

### 13.6 Numeric Conversion Validation

**What are we checking?** Whether measurement, count, sequence, and ordinal columns can be interpreted numerically using `pd.to_numeric(errors="coerce")`.

**Why are we checking it?** Mixed text in price, freight, dimensions, or payment values would break arithmetic and distort later statistics. Sequence and ordinal integers are included for type validation, not approved as continuous model features.

**What should we understand?** Original nulls are not conversion failures. This section does not assess negative, zero, or physically impossible values; those belong to Task 4.

In [24]:
NUMERIC_COLUMNS = {
    "Order Items": ["order_item_id", "price", "freight_value"],
    "Products": [
        "product_name_lenght",
        "product_description_lenght",
        "product_photos_qty",
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm",
    ],
    "Payments": [
        "payment_sequential",
        "payment_installments",
        "payment_value",
    ],
    "Reviews": ["review_score"],
    "Geolocation": ["geolocation_lat", "geolocation_lng"],
}


def numeric_conversion_profile(
    dataset_name: str, dataframe: pd.DataFrame, column: str
) -> tuple[dict, pd.DataFrame]:
    '''Test numeric parsing without treating original nulls as failures.'''
    original = dataframe[column]
    converted = pd.to_numeric(original, errors="coerce")
    original_non_null = original.notna()
    failure_mask = original_non_null & converted.isna()
    non_null_count = int(original_non_null.sum())
    successful_count = int((original_non_null & converted.notna()).sum())

    summary = {
        "Dataset": dataset_name,
        "Column": column,
        "Current Type": str(original.dtype),
        "Original Non-Null": non_null_count,
        "Successful Conversions": successful_count,
        "Conversion Failures": int(failure_mask.sum()),
        "Success Percentage": (
            successful_count / non_null_count * 100 if non_null_count else 100.0
        ),
    }
    failures = pd.DataFrame(
        {
            "Dataset": dataset_name,
            "Column": column,
            "Unparseable Value": original.loc[failure_mask]
            .astype("string")
            .unique()[:5],
        }
    )
    return summary, failures


numeric_summaries = []
numeric_failure_samples = []
for dataset_name, columns in NUMERIC_COLUMNS.items():
    for column in columns:
        summary, failures = numeric_conversion_profile(
            dataset_name, raw_datasets[dataset_name], column
        )
        numeric_summaries.append(summary)
        numeric_failure_samples.append(failures)

numeric_conversion_summary = pd.DataFrame(numeric_summaries)
numeric_conversion_failures = pd.concat(
    numeric_failure_samples, ignore_index=True
)
display(numeric_conversion_summary)
display(numeric_conversion_failures)

,Dataset,Column,Current Type,Original Non-Null,Successful Conversions,Conversion Failures,Success Percentage
0,Order Items,order_item_id,int64,112650,112650,0,100.0000
1,Order Items,price,float64,112650,112650,0,100.0000
2,Order Items,freight_value,float64,112650,112650,0,100.0000
3,Products,product_name_lenght,float64,32341,32341,0,100.0000
4,Products,product_description_lenght,float64,32341,32341,0,100.0000
5,Products,product_photos_qty,float64,32341,32341,0,100.0000
6,Products,product_weight_g,float64,32949,32949,0,100.0000
7,Products,product_length_cm,float64,32949,32949,0,100.0000
8,Products,product_height_cm,float64,32949,32949,0,100.0000
9,Products,product_width_cm,float64,32949,32949,0,100.0000


,Dataset,Column,Unparseable Value


**Interpretation guidance:** Successful numeric conversion establishes syntactic compatibility only. It does not authorize treating identifiers or ordinal categories as continuous quantities, and it does not establish that a numeric value is plausible.

### 13.7 Identifier and Categorical Type Validation

**What are we checking?** Storage and cardinality for important identifiers and categories, plus the effect of numeric ZIP parsing on leading zeros.

**Why are we checking it?** Opaque IDs should support exact matching, categories require later encoding rather than arithmetic, and ZIP prefixes are geographic codes whose formatting can be damaged by integer inference.

**What should we understand?** Storage type, semantic type, and eventual ML representation can differ legitimately. This task documents those distinctions but does not encode categories or create features.

In [25]:
IDENTIFIER_AND_CATEGORY_COLUMNS = {
    "Orders": ["order_id", "customer_id", "order_status"],
    "Order Items": ["order_id", "order_item_id", "product_id", "seller_id"],
    "Products": ["product_id", "product_category_name"],
    "Customers": [
        "customer_id",
        "customer_unique_id",
        "customer_zip_code_prefix",
        "customer_city",
        "customer_state",
    ],
    "Sellers": [
        "seller_id",
        "seller_zip_code_prefix",
        "seller_city",
        "seller_state",
    ],
    "Payments": ["order_id", "payment_sequential", "payment_type"],
    "Reviews": ["review_id", "order_id", "review_score"],
    "Geolocation": [
        "geolocation_zip_code_prefix",
        "geolocation_city",
        "geolocation_state",
    ],
    "Category Translation": [
        "product_category_name",
        "product_category_name_english",
    ],
}

identifier_category_rows = []
for dataset_name, columns in IDENTIFIER_AND_CATEGORY_COLUMNS.items():
    for column in columns:
        series = raw_datasets[dataset_name][column]
        non_null_values = series.loc[series.notna()]
        expected_type, semantic_role = EXPECTED_COLUMN_TYPES[dataset_name][column]
        identifier_category_rows.append(
            {
                "Dataset": dataset_name,
                "Column": column,
                "Current Type": str(series.dtype),
                "Expected Type": expected_type,
                "Semantic Role": semantic_role,
                "Distinct Non-Null Values": int(series.nunique()),
                "Example": non_null_values.iloc[0] if len(non_null_values) else None,
                "ML Interpretation": (
                    "Join/group key; not a continuous feature"
                    if semantic_role == "Identifier"
                    else "Geographic code; not a continuous feature"
                    if semantic_role == "Geographic identifier"
                    else "Requires later categorical/ordinal treatment if approved"
                ),
            }
        )

identifier_category_summary = pd.DataFrame(identifier_category_rows)
display(identifier_category_summary)

,Dataset,Column,Current Type,Expected Type,Semantic Role,Distinct Non-Null Values,Example,ML Interpretation
0,Orders,order_id,str,String,Identifier,99441,e481f51cbdc54678b7cc49136f2d6af7,Join/group key; not a continuous feature
1,Orders,customer_id,str,String,Identifier,99441,9ef432eb6251297304e76186b10a928d,Join/group key; not a continuous feature
2,Orders,order_status,str,Category,Categorical,8,delivered,Requires later categorical/ordinal treatment if approved
3,Order Items,order_id,str,String,Identifier,98666,00010242fe8c5a6d1ba2dd792cb16214,Join/group key; not a continuous feature
4,Order Items,order_item_id,int64,Integer sequence,Identifier,21,1,Join/group key; not a continuous feature
5,Order Items,product_id,str,String,Identifier,32951,4244733e06e7ecb4970a6e2683c13e61,Join/group key; not a continuous feature
6,Order Items,seller_id,str,String,Identifier,3095,48436dade18ac8b2bce089ec2a041202,Join/group key; not a continuous feature
7,Products,product_id,str,String,Identifier,32951,1e9e8ef04dbcff4541ed26657ea517e5,Join/group key; not a continuous feature
8,Products,product_category_name,str,Category,Categorical,73,perfumaria,Requires later categorical/ordinal treatment if approved
9,Customers,customer_id,str,String,Identifier,99441,06b8999e2fba1a1fbc88172c00ba8bc7,Join/group key; not a continuous feature


In [26]:
ZIP_COLUMNS = {
    "Customers": "customer_zip_code_prefix",
    "Sellers": "seller_zip_code_prefix",
    "Geolocation": "geolocation_zip_code_prefix",
}

zip_validation_rows = []
for dataset_name, column in ZIP_COLUMNS.items():
    # Reloading as literal text exposes formatting that integer inference loses.
    raw_text = load_raw_text_csv(DATASET_FILES[dataset_name])[column]
    inferred = raw_datasets[dataset_name][column]
    zip_validation_rows.append(
        {
            "Dataset": dataset_name,
            "Column": column,
            "Inferred Pandas Type": str(inferred.dtype),
            "Raw Values Starting With Zero": int(raw_text.str.startswith("0").sum()),
            "Minimum Raw Width": int(raw_text.str.len().min()),
            "Maximum Raw Width": int(raw_text.str.len().max()),
            "String Conversion Recommended": pd.api.types.is_numeric_dtype(
                inferred.dtype
            ),
        }
    )

zip_type_validation = pd.DataFrame(zip_validation_rows)
display(zip_type_validation)

,Dataset,Column,Inferred Pandas Type,Raw Values Starting With Zero,Minimum Raw Width,Maximum Raw Width,String Conversion Recommended
0,Customers,customer_zip_code_prefix,int64,23995,5,5,True
1,Sellers,seller_zip_code_prefix,int64,1027,5,5,True
2,Geolocation,geolocation_zip_code_prefix,int64,245733,5,5,True


**Interpretation guidance:** If raw ZIP prefixes start with zero while the inferred column is numeric, the numeric values have lost display width even though the underlying prefix can often be recovered by zero-padding. Later preparation should load or convert ZIP prefixes as fixed-width strings before joins. They must not be treated as continuous numerical features.

### 13.8 ML Impact Assessment

| Type issue | Classification impact | Regression impact | Purchase-time and preparation rule |
|---|---|---|---|
| Unparseable actual or estimated delivery timestamp | Prevents reliable `is_late` construction | Actual-delivery failures also prevent duration construction | Target-only fields must remain outside features; quantify label exclusions |
| Unparseable purchase timestamp | Breaks chronological splitting and purchase-calendar interpretation | Prevents delivery-duration calculation | Purchase timestamp is the prediction boundary; validate before use |
| Numeric measurement stored as mixed text | Can break preprocessing or silently create missing features | Can distort arithmetic and model inputs | Convert through a validated, train-consistent preparation step |
| ZIP prefix treated as continuous numeric | Creates meaningless numeric distance/order | Same risk for duration modeling | Preserve as a geographic code; use only an approved categorical/static lookup representation |
| Opaque identifier used as a numeric/model feature | Encourages memorization rather than generalizable signal | Same risk and unstable extrapolation | Retain for joins, grouping, lineage, and split integrity—not as a raw feature |
| Category left as arbitrary raw strings | Most estimators cannot consume it directly | Same limitation | Encode later using training-fitted preprocessing only |
| Post-purchase timestamp or review type made model-ready | Can make leakage easier to overlook | Can leak the observed outcome | Type compatibility does not override the purchase-time availability policy |

Correct dtypes are necessary but not sufficient for feature approval. Every future feature still requires purchase-time availability, order-grain aggregation, leakage controls, and training-serving parity.

### 13.9 Findings and Recommendations

**What are we checking?** A consolidated result for every column, combining dtype inspection with temporary datetime/numeric conversion evidence and ZIP-format evidence.

**Why are we checking it?** Recommendations should be based on actual conversion results rather than assumptions about `object`, `str`, or numeric dtypes.

**What should we understand?** “Conversion Recommended Later” records preparation work; it does not mutate data here. “Suspicious / Needs Investigation” is reserved for actual conversion failures or incompatible storage.

In [27]:
datetime_result_index = datetime_conversion_summary.set_index(
    ["Dataset", "Column"]
)
numeric_result_index = numeric_conversion_summary.set_index(
    ["Dataset", "Column"]
)
zip_result_index = zip_type_validation.set_index(["Dataset", "Column"])


def ml_type_impact(semantic_role: str, dataset_name: str, column: str) -> str:
    '''Describe the downstream risk without approving a model feature.'''
    if semantic_role == "Datetime":
        if column in {
            "order_purchase_timestamp",
            "order_delivered_customer_date",
            "order_estimated_delivery_date",
        }:
            return "Target eligibility, duration calculation, or chronological splitting"
        return "Audit/retrospective timing; post-purchase fields remain excluded"
    if semantic_role in {"Numeric measurement", "Count"}:
        return "Potential feature arithmetic or statistical analysis"
    if semantic_role == "Geographic identifier":
        return "Join accuracy and prevention of meaningless continuous treatment"
    if semantic_role == "Identifier":
        return "Join, grouping, grain, and leakage control; not a raw feature"
    if dataset_name == "Reviews":
        return "Retrospective only; prohibited as a purchase-time feature"
    return "Potential categorical handling if later approved"


def recommended_type_action(
    semantic_role: str, validation_result: str
) -> str:
    '''Map evidence to a future, non-destructive handling recommendation.'''
    if validation_result == "Suspicious / Needs Investigation":
        return "Investigate unparseable or incompatible values before preparation"
    if semantic_role == "Datetime":
        return "Parse in a controlled preparation step; retain original audit evidence"
    if semantic_role == "Geographic identifier":
        return "Load/convert as a fixed-width string code before geographic joins"
    if semantic_role == "Identifier":
        return "Retain for joins and lineage; do not treat as a continuous feature"
    if semantic_role in {"Categorical", "Ordinal categorical"}:
        return "Retain semantics; encode later only if purchase-time approved"
    if semantic_role == "Text":
        return "Retain for retrospective analysis; exclude review text from model inputs"
    return "Retain current representation; validate again during preparation"


data_type_findings_rows = []
for row in dtype_comparison.to_dict(orient="records"):
    key = (row["Dataset"], row["Column"])
    validation_result = row["Initial Validation Result"]
    evidence = f"Pandas inferred {row['Pandas Type']}."

    if row["Semantic Role"] == "Datetime":
        conversion = datetime_result_index.loc[key]
        failures = int(conversion["Conversion Failures"])
        validation_result = (
            "Suspicious / Needs Investigation"
            if failures
            else "Correct / Compatible"
            if pd.api.types.is_datetime64_any_dtype(
                raw_datasets[row["Dataset"]][row["Column"]].dtype
            )
            else "Conversion Recommended Later"
        )
        evidence = (
            f"{int(conversion['Successful Conversions'])} of "
            f"{int(conversion['Original Non-Null'])} non-null values parsed; "
            f"{failures} failed."
        )
    elif row["Semantic Role"] in {
        "Numeric measurement",
        "Count",
        "Ordinal categorical",
    }:
        conversion = numeric_result_index.loc[key]
        failures = int(conversion["Conversion Failures"])
        validation_result = (
            "Suspicious / Needs Investigation"
            if failures
            else "Correct / Compatible"
        )
        evidence = (
            f"{int(conversion['Successful Conversions'])} of "
            f"{int(conversion['Original Non-Null'])} non-null values converted; "
            f"{failures} failed."
        )
    elif row["Semantic Role"] == "Geographic identifier":
        zip_result = zip_result_index.loc[key]
        leading_zero_count = int(zip_result["Raw Values Starting With Zero"])
        validation_result = "Conversion Recommended Later"
        evidence = (
            f"Raw text contains {leading_zero_count} values starting with zero; "
            f"Pandas inferred {zip_result['Inferred Pandas Type']}."
        )

    data_type_findings_rows.append(
        {
            "Dataset": DATASET_FILES[row["Dataset"]],
            "Column": row["Column"],
            "Current Type": row["Pandas Type"],
            "Expected Type": row["Expected Type"],
            "Semantic Role": row["Semantic Role"],
            "Validation Result": validation_result,
            "Evidence": evidence,
            "ML Impact": ml_type_impact(
                row["Semantic Role"], row["Dataset"], row["Column"]
            ),
            "Recommended Action": recommended_type_action(
                row["Semantic Role"], validation_result
            ),
        }
    )

data_type_findings = pd.DataFrame(data_type_findings_rows)
display(data_type_findings)
display(data_type_findings["Validation Result"].value_counts())

,Dataset,Column,Current Type,Expected Type,Semantic Role,Validation Result,Evidence,ML Impact,Recommended Action
0,olist_orders_dataset.csv,order_id,str,String,Identifier,Correct / Compatible,Pandas inferred str.,"Join, grouping, grain, and leakage control; not a raw feature",Retain for joins and lineage; do not treat as a continuous feature
1,olist_orders_dataset.csv,customer_id,str,String,Identifier,Correct / Compatible,Pandas inferred str.,"Join, grouping, grain, and leakage control; not a raw feature",Retain for joins and lineage; do not treat as a continuous feature
2,olist_orders_dataset.csv,order_status,str,Category,Categorical,Correct / Compatible,Pandas inferred str.,Potential categorical handling if later approved,Retain semantics; encode later only if purchase-time approved
3,olist_orders_dataset.csv,order_purchase_timestamp,str,Datetime,Datetime,Conversion Recommended Later,99441 of 99441 non-null values parsed; 0 failed.,"Target eligibility, duration calculation, or chronological splitting",Parse in a controlled preparation step; retain original audit evidence
4,olist_orders_dataset.csv,order_approved_at,str,Datetime,Datetime,Conversion Recommended Later,99281 of 99281 non-null values parsed; 0 failed.,Audit/retrospective timing; post-purchase fields remain excluded,Parse in a controlled preparation step; retain original audit evidence
5,olist_orders_dataset.csv,order_delivered_carrier_date,str,Datetime,Datetime,Conversion Recommended Later,97658 of 97658 non-null values parsed; 0 failed.,Audit/retrospective timing; post-purchase fields remain excluded,Parse in a controlled preparation step; retain original audit evidence
6,olist_orders_dataset.csv,order_delivered_customer_date,str,Datetime,Datetime,Conversion Recommended Later,96476 of 96476 non-null values parsed; 0 failed.,"Target eligibility, duration calculation, or chronological splitting",Parse in a controlled preparation step; retain original audit evidence
7,olist_orders_dataset.csv,order_estimated_delivery_date,str,Datetime,Datetime,Conversion Recommended Later,99441 of 99441 non-null values parsed; 0 failed.,"Target eligibility, duration calculation, or chronological splitting",Parse in a controlled preparation step; retain original audit evidence
8,olist_order_items_dataset.csv,order_id,str,String,Identifier,Correct / Compatible,Pandas inferred str.,"Join, grouping, grain, and leakage control; not a raw feature",Retain for joins and lineage; do not treat as a continuous feature
9,olist_order_items_dataset.csv,order_item_id,int64,Integer sequence,Identifier,Correct / Compatible,Pandas inferred int64.,"Join, grouping, grain, and leakage control; not a raw feature",Retain for joins and lineage; do not treat as a continuous feature


Validation Result
Correct / Compatible            41
Conversion Recommended Later    11
Name: count, dtype: int64

### 13.10 Learning Questions

These questions are intentionally unanswered.

1. What is the difference between a Pandas dtype and a semantic data type?
2. Why can datetime columns initially appear as `object`?
3. Why should ZIP codes not be treated as continuous numeric features?
4. What does `errors="coerce"` do during conversion?
5. How can we distinguish original missing values from conversion failures?
6. Why can an integer column represent a category instead of a numerical measurement?
7. How can incorrect data types affect regression and classification?
8. Why should we validate conversions before permanently changing data types?

## 14. Invalid Value Validation

### 14.1 Objectives and Validation Principles

This section implements **Task 4 only: invalid-value validation** across all nine Olist datasets. It asks whether present values satisfy evidence-backed structural, domain, or business rules; it does not repeat missingness, duplicate, dtype, timestamp-ordering, referential-integrity, or outlier work.

Three result classes keep evidence and uncertainty separate:

| Classification | Meaning |
|---|---|
| Valid | The evaluated value satisfies this particular rule; this does not confirm real-world factual accuracy. |
| Invalid | A present value clearly violates a documented structural or domain constraint. |
| Needs Investigation | A present value is unusual under an explicit screening assumption but may be legitimate. |

Original nulls are excluded from every denominator. A missing value was assessed in Task 1 and is not relabeled as invalid here. All masks and summaries are temporary: no source DataFrame or CSV is changed.

Business-rule validity is also separate from feature eligibility. Reviews and lifecycle outcomes remain retrospective even when their values are valid, and every future feature still has to satisfy the purchase-time boundary.

### 14.2 Define Dataset-Specific Business Rules

**What are we checking?** A compact rule specification that states the expected condition, evidence, severity if violated, and possible ML relevance.

**Why does it matter?** A validation result is defensible only when the rule is stated before interpreting the count. Proven constraints (such as the 1–5 review scale) must not be confused with assumptions (such as an approximate Brazil boundary).

**How should the result be interpreted?** `Violation Classification` describes how a failing *non-null* value will be labeled. A rule with zero observed failures will appear as `Valid` in the calculated findings table, while retaining its intended failure class here.

In [28]:
BRAZIL_STATE_CODES = {
    "AC", "AL", "AP", "AM", "BA", "CE", "DF", "ES", "GO",
    "MA", "MT", "MS", "MG", "PA", "PB", "PR", "PE", "PI",
    "RJ", "RN", "RS", "RO", "RR", "SC", "SP", "SE", "TO",
}
DOCUMENTED_ORDER_STATUSES = {
    "created", "approved", "invoiced", "processing", "shipped",
    "delivered", "unavailable", "canceled",
}
DOCUMENTED_PAYMENT_TYPES = {
    "credit_card", "boleto", "voucher", "debit_card", "not_defined",
}
HEX_ID_PATTERN = r"^[0-9a-f]{32}$"
SNAKE_CASE_PATTERN = r"^[a-z0-9]+(?:_[a-z0-9]+)*$"

rule_specifications = pd.DataFrame([
    {"Dataset": "Orders", "Column": "order_status", "Rule Description": "Status uses the documented Olist vocabulary", "Expected Condition": "Value is in the eight documented statuses", "Rule Justification": "DATA_DICTIONARY.md documents the source vocabulary", "Violation Classification": "Invalid", "Potential ML Relevance": "Unexpected outcome categories can corrupt retrospective cohorts; status is post-purchase"},
    {"Dataset": "Orders", "Column": "order_id, customer_id", "Rule Description": "Opaque identifiers use the observed source representation", "Expected Condition": "Exactly 32 lowercase hexadecimal characters", "Rule Justification": "Source schema and all documented examples use this representation", "Violation Classification": "Invalid", "Potential ML Relevance": "Malformed join keys can disconnect orders from related records"},
    {"Dataset": "Order Items", "Column": "price, freight_value", "Rule Description": "Currency amounts are nonnegative", "Expected Condition": "Value >= 0", "Rule Justification": "Negative charged amounts contradict the documented amount semantics", "Violation Classification": "Invalid", "Potential ML Relevance": "Invalid monetary inputs distort order-level aggregates"},
    {"Dataset": "Order Items", "Column": "price, freight_value", "Rule Description": "Zero amounts receive contextual review", "Expected Condition": "Value > 0 for the screening rule", "Rule Justification": "Free items or free freight can be legitimate; zero is not proven corrupt", "Violation Classification": "Needs Investigation", "Potential ML Relevance": "Zero-heavy features can change ratios and segment comparisons"},
    {"Dataset": "Order Items", "Column": "order_item_id", "Rule Description": "Item sequence is positive", "Expected Condition": "Value > 0", "Rule Justification": "The field is a one-based within-order sequence", "Violation Classification": "Invalid", "Potential ML Relevance": "Invalid sequence values undermine item grain and counts"},
    {"Dataset": "Products", "Column": "physical measurements", "Rule Description": "Physical measurements are not negative", "Expected Condition": "Value >= 0", "Rule Justification": "Negative mass or dimensions are physically impossible", "Violation Classification": "Invalid", "Potential ML Relevance": "Impossible measurements distort shipment-size signals"},
    {"Dataset": "Products", "Column": "physical measurements", "Rule Description": "Zero measurements receive contextual review", "Expected Condition": "Value > 0 for the screening rule", "Rule Justification": "Zero can be a sentinel or unusual catalog record but is not automatically proven wrong", "Violation Classification": "Needs Investigation", "Potential ML Relevance": "Zeros can distort volume, weight, and downstream predictions"},
    {"Dataset": "Products", "Column": "metadata counts and lengths", "Rule Description": "Counts and text lengths are not negative", "Expected Condition": "Value >= 0", "Rule Justification": "Counts and character lengths cannot be negative", "Violation Classification": "Invalid", "Potential ML Relevance": "Invalid catalog metadata distorts numeric distributions"},
    {"Dataset": "Products", "Column": "product_photos_qty", "Rule Description": "Zero photo counts receive contextual review", "Expected Condition": "Value > 0 for the screening rule", "Rule Justification": "A zero-photo listing is unusual but may exist", "Violation Classification": "Needs Investigation", "Potential ML Relevance": "May represent catalog completeness rather than product behavior"},
    {"Dataset": "Customers / Sellers / Geolocation", "Column": "state", "Rule Description": "State is a recognized Brazilian UF abbreviation", "Expected Condition": "Value is one of 27 official two-letter codes", "Rule Justification": "Brazil has 26 states plus the Federal District", "Violation Classification": "Invalid", "Potential ML Relevance": "Invalid regions corrupt geographic grouping"},
    {"Dataset": "Customers / Sellers / Geolocation", "Column": "ZIP prefix", "Rule Description": "Raw ZIP prefix has five digits", "Expected Condition": "Raw text matches exactly five digits", "Rule Justification": "The source field is a five-digit postal prefix; it is a code, not a numeric range", "Violation Classification": "Invalid", "Potential ML Relevance": "Malformed prefixes break geographic lookup and subgroup analysis"},
    {"Dataset": "Payments", "Column": "payment_value", "Rule Description": "Payment value is nonnegative", "Expected Condition": "Value >= 0", "Rule Justification": "Negative payment amounts contradict the documented payment-row amount", "Violation Classification": "Invalid", "Potential ML Relevance": "Invalid values distort financial aggregates"},
    {"Dataset": "Payments", "Column": "payment_value", "Rule Description": "Zero payments receive contextual review", "Expected Condition": "Value > 0 for the screening rule", "Rule Justification": "Zero records may reflect voucher/accounting behavior and require context", "Violation Classification": "Needs Investigation", "Potential ML Relevance": "Zeros can distort totals and payment summaries"},
    {"Dataset": "Payments", "Column": "payment_installments", "Rule Description": "Installments are not negative; zero is reviewed", "Expected Condition": "Value >= 0; zero is screened separately", "Rule Justification": "Negative counts are impossible; zero exists in the source and is ambiguous", "Violation Classification": "Invalid / Needs Investigation", "Potential ML Relevance": "Invalid counts distort payment structure"},
    {"Dataset": "Payments", "Column": "payment_sequential", "Rule Description": "Payment sequence is positive", "Expected Condition": "Value > 0", "Rule Justification": "The field is a one-based within-order sequence", "Violation Classification": "Invalid", "Potential ML Relevance": "Invalid sequences undermine payment grain"},
    {"Dataset": "Payments", "Column": "payment_type", "Rule Description": "Payment type uses documented categories", "Expected Condition": "Value is in the five documented categories, including not_defined", "Rule Justification": "DATA_DICTIONARY.md records not_defined as a literal category, not a null", "Violation Classification": "Invalid", "Potential ML Relevance": "Unexpected categories can break later encoders"},
    {"Dataset": "Reviews", "Column": "review_score", "Rule Description": "Review score is on the documented 1–5 scale", "Expected Condition": "Value is in {1, 2, 3, 4, 5}", "Rule Justification": "Published scale documented in DATA_DICTIONARY.md", "Violation Classification": "Invalid", "Potential ML Relevance": "Retrospective analysis only; prohibited as a purchase-time feature"},
    {"Dataset": "Reviews", "Column": "review text", "Rule Description": "Present comment text is not whitespace-only", "Expected Condition": "Non-empty raw text contains a non-whitespace character", "Rule Justification": "Absent optional comments are valid; whitespace-only text is structurally empty", "Violation Classification": "Invalid", "Potential ML Relevance": "Retrospective text quality only; review data is post-purchase"},
    {"Dataset": "Geolocation", "Column": "latitude / longitude", "Rule Description": "Coordinates are finite and inside global bounds", "Expected Condition": "Finite latitude in [-90, 90] and longitude in [-180, 180]", "Rule Justification": "Global coordinate-system constraints", "Violation Classification": "Invalid", "Potential ML Relevance": "Impossible coordinates corrupt geographic distance features"},
    {"Dataset": "Geolocation", "Column": "latitude / longitude", "Rule Description": "Coordinates fall inside an approximate Brazil boundary", "Expected Condition": "Latitude [-34, 6], longitude [-74, -32]", "Rule Justification": "Broad screening box from project data dictionary; not an exact national border", "Violation Classification": "Needs Investigation", "Potential ML Relevance": "Suspicious coordinates can distort routes and regional comparisons"},
    {"Dataset": "Category Translation", "Column": "both category columns", "Rule Description": "Present category labels are nonblank and structurally consistent", "Expected Condition": "Contains non-whitespace text; snake_case is a source-format screen", "Rule Justification": "Blank labels cannot identify a category; snake_case reflects the documented source representation", "Violation Classification": "Invalid / Needs Investigation", "Potential ML Relevance": "Malformed labels can fragment category encoding"},
])
display(rule_specifications)

,Dataset,Column,Rule Description,Expected Condition,Rule Justification,Violation Classification,Potential ML Relevance
0,Orders,order_status,Status uses the documented Olist vocabulary,Value is in the eight documented statuses,DATA_DICTIONARY.md documents the source vocabulary,Invalid,Unexpected outcome categories can corrupt retrospective cohorts; status is post-purchase
1,Orders,"order_id, customer_id",Opaque identifiers use the observed source representation,Exactly 32 lowercase hexadecimal characters,Source schema and all documented examples use this representation,Invalid,Malformed join keys can disconnect orders from related records
2,Order Items,"price, freight_value",Currency amounts are nonnegative,Value >= 0,Negative charged amounts contradict the documented amount semantics,Invalid,Invalid monetary inputs distort order-level aggregates
3,Order Items,"price, freight_value",Zero amounts receive contextual review,Value > 0 for the screening rule,Free items or free freight can be legitimate; zero is not proven corrupt,Needs Investigation,Zero-heavy features can change ratios and segment comparisons
4,Order Items,order_item_id,Item sequence is positive,Value > 0,The field is a one-based within-order sequence,Invalid,Invalid sequence values undermine item grain and counts
5,Products,physical measurements,Physical measurements are not negative,Value >= 0,Negative mass or dimensions are physically impossible,Invalid,Impossible measurements distort shipment-size signals
6,Products,physical measurements,Zero measurements receive contextual review,Value > 0 for the screening rule,Zero can be a sentinel or unusual catalog record but is not automatically proven wrong,Needs Investigation,"Zeros can distort volume, weight, and downstream predictions"
7,Products,metadata counts and lengths,Counts and text lengths are not negative,Value >= 0,Counts and character lengths cannot be negative,Invalid,Invalid catalog metadata distorts numeric distributions
8,Products,product_photos_qty,Zero photo counts receive contextual review,Value > 0 for the screening rule,A zero-photo listing is unusual but may exist,Needs Investigation,May represent catalog completeness rather than product behavior
9,Customers / Sellers / Geolocation,state,State is a recognized Brazilian UF abbreviation,Value is one of 27 official two-letter codes,Brazil has 26 states plus the Federal District,Invalid,Invalid regions corrupt geographic grouping


In [29]:
invalid_value_findings_rows = []
invalid_value_samples = []


def record_rule(
    dataset_name: str,
    dataframe: pd.DataFrame,
    column_label: str,
    rule: str,
    applicable_mask: pd.Series,
    violation_mask: pd.Series,
    violation_classification: str,
    evidence: str,
    ml_impact: str,
    recommended_action: str,
    sample_columns: list[str] | None = None,
) -> None:
    '''Record one non-destructive rule result with a non-null denominator.

    `applicable_mask` identifies present values eligible for the rule. The
    violation mask is intersected with it so Task 1 missingness can never be
    relabeled as an invalid value. Small samples support interpretation without
    printing full exception sets.
    '''
    applicable = applicable_mask.fillna(False).astype(bool)
    affected = (violation_mask.fillna(False).astype(bool) & applicable)
    applicable_count = int(applicable.sum())
    affected_count = int(affected.sum())
    invalid_value_findings_rows.append({
        "Dataset": DATASET_FILES[dataset_name],
        "Column": column_label,
        "Validation Rule": rule,
        "Applicable Records": applicable_count,
        "Affected Records": affected_count,
        "Affected Percentage": (
            affected_count / applicable_count * 100 if applicable_count else 0.0
        ),
        "Classification": violation_classification if affected_count else "Valid",
        "Evidence": evidence,
        "ML Impact": ml_impact,
        "Recommended Action": recommended_action,
    })
    if affected_count and sample_columns:
        sample = dataframe.loc[affected, sample_columns].head(5).copy()
        sample.insert(0, "Rule", rule)
        sample.insert(0, "Dataset", dataset_name)
        invalid_value_samples.append(sample)


def record_series_rule(
    dataset_name: str,
    dataframe: pd.DataFrame,
    column: str,
    rule: str,
    violation_mask: pd.Series,
    violation_classification: str,
    evidence: str,
    ml_impact: str,
    recommended_action: str,
    sample_columns: list[str] | None = None,
) -> None:
    '''Record a single-column rule using its non-null values as denominator.'''
    record_rule(
        dataset_name, dataframe, column, rule, dataframe[column].notna(),
        violation_mask, violation_classification, evidence, ml_impact,
        recommended_action, sample_columns or [column],
    )

### 14.3 Orders Validation

**What are we checking?** The documented status vocabulary and the structural representation of `order_id` and `customer_id`.

**Why does it matter?** Unexpected statuses can invalidate retrospective cohort logic, while malformed identifiers can break exact joins. Uniqueness was already assessed in Task 2 and is not repeated.

**How should the result be interpreted?** The 32-character lowercase hexadecimal format is a source-schema representation check, not proof that an identifier points to the correct real-world entity. Status is a post-purchase outcome and remains ineligible as a purchase-time feature.

In [30]:
orders_raw = raw_datasets["Orders"]
record_series_rule("Orders", orders_raw, "order_status", "Documented order-status vocabulary", ~orders_raw["order_status"].isin(DOCUMENTED_ORDER_STATUSES), "Invalid", "Compared with the eight statuses documented in DATA_DICTIONARY.md.", "Unexpected statuses can corrupt retrospective cohorts; status itself would leak post-purchase information.", "Investigate upstream category changes; do not map silently.", ["order_id", "order_status"])
for column in ["order_id", "customer_id"]:
    record_series_rule("Orders", orders_raw, column, "Exactly 32 lowercase hexadecimal characters", ~orders_raw[column].astype("string").str.fullmatch(HEX_ID_PATTERN, na=False), "Invalid", "Objective format check against the source schema representation.", "Malformed keys can break exact joins and detach order context.", "Quarantine malformed keys during later preparation and trace the source.", ["order_id", "customer_id"])

orders_validation = pd.DataFrame(invalid_value_findings_rows[-3:])
display(orders_validation)

,Dataset,Column,Validation Rule,Applicable Records,Affected Records,Affected Percentage,Classification,Evidence,ML Impact,Recommended Action
0,olist_orders_dataset.csv,order_status,Documented order-status vocabulary,99441,0,0.0000,Valid,Compared with the eight statuses documented in DATA_DICTIONARY.md.,Unexpected statuses can corrupt retrospective cohorts; status itself would leak post-purchase in...,Investigate upstream category changes; do not map silently.
1,olist_orders_dataset.csv,order_id,Exactly 32 lowercase hexadecimal characters,99441,0,0.0000,Valid,Objective format check against the source schema representation.,Malformed keys can break exact joins and detach order context.,Quarantine malformed keys during later preparation and trace the source.
2,olist_orders_dataset.csv,customer_id,Exactly 32 lowercase hexadecimal characters,99441,0,0.0000,Valid,Objective format check against the source schema representation.,Malformed keys can break exact joins and detach order context.,Quarantine malformed keys during later preparation and trace the source.


### 14.4 Order Items Validation

**What are we checking?** Negative and zero price/freight values, positive item sequence numbers, and relevant opaque identifier formats.

**Why does it matter?** Currency values may become purchase-time aggregates, and sequence/key defects can undermine item grain.

**How should the result be interpreted?** Negative amounts and non-positive sequences violate their semantics. Zero price is suspicious, while zero freight can represent a legitimate free-shipping arrangement; neither zero screen performs cleaning or outlier analysis.

In [31]:
items_raw = raw_datasets["Order Items"]
for column in ["price", "freight_value"]:
    record_series_rule("Order Items", items_raw, column, f"{column} must not be negative", items_raw[column] < 0, "Invalid", "Currency amounts cannot be negative under the documented charge semantics.", "Invalid monetary values distort order aggregates for both objectives.", "Investigate source records before aggregation; do not clip silently.", ["order_id", "order_item_id", column])
record_series_rule("Order Items", items_raw, "price", "Zero price requires business-context review", items_raw["price"].eq(0), "Needs Investigation", "A free/promotional item is possible, so zero is not automatically invalid.", "Can alter order-value statistics and subgroup comparisons.", "Confirm promotion/accounting semantics before later treatment.", ["order_id", "order_item_id", "price"])
record_series_rule("Order Items", items_raw, "freight_value", "Zero freight is a plausible free-shipping case", items_raw["freight_value"].eq(0), "Needs Investigation", "Zero freight may be a legitimate commercial arrangement.", "Can affect freight ratios and delivery-risk segment comparisons.", "Retain and document unless external evidence establishes an error.", ["order_id", "order_item_id", "freight_value"])
record_series_rule("Order Items", items_raw, "order_item_id", "Item sequence must be positive", items_raw["order_item_id"] <= 0, "Invalid", "A within-order item sequence is one-based.", "Invalid sequences undermine item counts and grain.", "Investigate and quarantine before order-level aggregation.", ["order_id", "order_item_id"])
for column in ["order_id", "product_id", "seller_id"]:
    record_series_rule("Order Items", items_raw, column, "Exactly 32 lowercase hexadecimal characters", ~items_raw[column].astype("string").str.fullmatch(HEX_ID_PATTERN, na=False), "Invalid", "Objective source identifier representation.", "Malformed keys can break later joins and order aggregation.", "Trace malformed keys to source; do not repair by guessing.", ["order_id", "order_item_id", column])

order_items_validation = pd.DataFrame(invalid_value_findings_rows[-8:])
display(order_items_validation)

,Dataset,Column,Validation Rule,Applicable Records,Affected Records,Affected Percentage,Classification,Evidence,ML Impact,Recommended Action
0,olist_order_items_dataset.csv,price,price must not be negative,112650,0,0.0000,Valid,Currency amounts cannot be negative under the documented charge semantics.,Invalid monetary values distort order aggregates for both objectives.,Investigate source records before aggregation; do not clip silently.
1,olist_order_items_dataset.csv,freight_value,freight_value must not be negative,112650,0,0.0000,Valid,Currency amounts cannot be negative under the documented charge semantics.,Invalid monetary values distort order aggregates for both objectives.,Investigate source records before aggregation; do not clip silently.
2,olist_order_items_dataset.csv,price,Zero price requires business-context review,112650,0,0.0000,Valid,"A free/promotional item is possible, so zero is not automatically invalid.",Can alter order-value statistics and subgroup comparisons.,Confirm promotion/accounting semantics before later treatment.
3,olist_order_items_dataset.csv,freight_value,Zero freight is a plausible free-shipping case,112650,383,0.3400,Needs Investigation,Zero freight may be a legitimate commercial arrangement.,Can affect freight ratios and delivery-risk segment comparisons.,Retain and document unless external evidence establishes an error.
4,olist_order_items_dataset.csv,order_item_id,Item sequence must be positive,112650,0,0.0000,Valid,A within-order item sequence is one-based.,Invalid sequences undermine item counts and grain.,Investigate and quarantine before order-level aggregation.
5,olist_order_items_dataset.csv,order_id,Exactly 32 lowercase hexadecimal characters,112650,0,0.0000,Valid,Objective source identifier representation.,Malformed keys can break later joins and order aggregation.,Trace malformed keys to source; do not repair by guessing.
6,olist_order_items_dataset.csv,product_id,Exactly 32 lowercase hexadecimal characters,112650,0,0.0000,Valid,Objective source identifier representation.,Malformed keys can break later joins and order aggregation.,Trace malformed keys to source; do not repair by guessing.
7,olist_order_items_dataset.csv,seller_id,Exactly 32 lowercase hexadecimal characters,112650,0,0.0000,Valid,Objective source identifier representation.,Malformed keys can break later joins and order aggregation.,Trace malformed keys to source; do not repair by guessing.


### 14.5 Products Validation

**What are we checking?** Negative and zero physical measurements, negative metadata counts/text lengths, zero photo counts, and nonblank category structure for present categories.

**Why does it matter?** Weight and dimensions can influence delivery-duration and lateness models; invalid catalog counts can distort numerical distributions.

**How should the result be interpreted?** Negative measurements/counts are impossible. Zero measurements and zero photo counts are screened as suspicious because they may be sentinels or unusual catalog records. Missing descriptive fields remain Task 1 missingness, and large values are not classified as errors here.

In [32]:
products_raw = raw_datasets["Products"]
physical_columns = ["product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"]
metadata_numeric_columns = ["product_photos_qty", "product_name_lenght", "product_description_lenght"]
for column in physical_columns:
    record_series_rule("Products", products_raw, column, "Physical measurement must not be negative", products_raw[column] < 0, "Invalid", "Negative mass or dimensions are physically impossible.", "Impossible values distort size/weight distributions and predictions.", "Investigate source; later exclude or flag only under a documented preparation policy.", ["product_id", column])
    record_series_rule("Products", products_raw, column, "Zero physical measurement requires investigation", products_raw[column].eq(0), "Needs Investigation", "Zero may be a sentinel or unusual catalog record; missing values are excluded.", "Zero weight/dimensions can distort volume and shipment-complexity features.", "Validate catalog semantics and preserve an issue flag during later preparation.", ["product_id", column])
for column in metadata_numeric_columns:
    record_series_rule("Products", products_raw, column, "Count or text length must not be negative", products_raw[column] < 0, "Invalid", "Counts and character lengths cannot be negative.", "Invalid metadata changes feature distributions for either model.", "Investigate upstream catalog values; do not clip silently.", ["product_id", column])
record_series_rule("Products", products_raw, "product_photos_qty", "Zero photo count requires investigation", products_raw["product_photos_qty"].eq(0), "Needs Investigation", "A listing without photos is unusual but not impossible.", "May indicate catalog completeness rather than a direct delivery signal.", "Confirm meaning before deciding whether to retain or flag.", ["product_id", "product_photos_qty"])
product_category = products_raw["product_category_name"].astype("string")
record_series_rule("Products", products_raw, "product_category_name", "Present category must not be whitespace-only", product_category.str.strip().eq(""), "Invalid", "Whitespace-only text is structurally empty; actual nulls are excluded.", "Blank categories fragment or erase categorical grouping.", "Treat as an upstream malformed value, separately from missingness.", ["product_id", "product_category_name"])
record_series_rule("Products", products_raw, "product_category_name", "Present category follows observed snake_case structure", ~product_category.str.fullmatch(SNAKE_CASE_PATTERN, na=False), "Needs Investigation", "This is an observed source-format heuristic, not a fixed category vocabulary.", "Format drift can fragment later category encoding.", "Review new formats; do not reject new legitimate categories automatically.", ["product_id", "product_category_name"])

products_validation = pd.DataFrame(invalid_value_findings_rows[-14:])
display(products_validation)

,Dataset,Column,Validation Rule,Applicable Records,Affected Records,Affected Percentage,Classification,Evidence,ML Impact,Recommended Action
0,olist_products_dataset.csv,product_weight_g,Physical measurement must not be negative,32949,0,0.0000,Valid,Negative mass or dimensions are physically impossible.,Impossible values distort size/weight distributions and predictions.,Investigate source; later exclude or flag only under a documented preparation policy.
1,olist_products_dataset.csv,product_weight_g,Zero physical measurement requires investigation,32949,4,0.0121,Needs Investigation,Zero may be a sentinel or unusual catalog record; missing values are excluded.,Zero weight/dimensions can distort volume and shipment-complexity features.,Validate catalog semantics and preserve an issue flag during later preparation.
2,olist_products_dataset.csv,product_length_cm,Physical measurement must not be negative,32949,0,0.0000,Valid,Negative mass or dimensions are physically impossible.,Impossible values distort size/weight distributions and predictions.,Investigate source; later exclude or flag only under a documented preparation policy.
3,olist_products_dataset.csv,product_length_cm,Zero physical measurement requires investigation,32949,0,0.0000,Valid,Zero may be a sentinel or unusual catalog record; missing values are excluded.,Zero weight/dimensions can distort volume and shipment-complexity features.,Validate catalog semantics and preserve an issue flag during later preparation.
4,olist_products_dataset.csv,product_height_cm,Physical measurement must not be negative,32949,0,0.0000,Valid,Negative mass or dimensions are physically impossible.,Impossible values distort size/weight distributions and predictions.,Investigate source; later exclude or flag only under a documented preparation policy.
5,olist_products_dataset.csv,product_height_cm,Zero physical measurement requires investigation,32949,0,0.0000,Valid,Zero may be a sentinel or unusual catalog record; missing values are excluded.,Zero weight/dimensions can distort volume and shipment-complexity features.,Validate catalog semantics and preserve an issue flag during later preparation.
6,olist_products_dataset.csv,product_width_cm,Physical measurement must not be negative,32949,0,0.0000,Valid,Negative mass or dimensions are physically impossible.,Impossible values distort size/weight distributions and predictions.,Investigate source; later exclude or flag only under a documented preparation policy.
7,olist_products_dataset.csv,product_width_cm,Zero physical measurement requires investigation,32949,0,0.0000,Valid,Zero may be a sentinel or unusual catalog record; missing values are excluded.,Zero weight/dimensions can distort volume and shipment-complexity features.,Validate catalog semantics and preserve an issue flag during later preparation.
8,olist_products_dataset.csv,product_photos_qty,Count or text length must not be negative,32341,0,0.0000,Valid,Counts and character lengths cannot be negative.,Invalid metadata changes feature distributions for either model.,Investigate upstream catalog values; do not clip silently.
9,olist_products_dataset.csv,product_name_lenght,Count or text length must not be negative,32341,0,0.0000,Valid,Counts and character lengths cannot be negative.,Invalid metadata changes feature distributions for either model.,Investigate upstream catalog values; do not clip silently.


### 14.6 Customers and Sellers Validation

**What are we checking?** Recognized Brazilian state abbreviations, raw five-digit ZIP-prefix structure, and opaque identifier representation.

**Why does it matter?** Geographic codes drive regional comparisons and potential static lookups. Reading ZIPs as integers erases leading-zero display width and can produce a false validation failure.

**How should the result be interpreted?** ZIP prefixes are reloaded as literal text and checked with a five-digit regular expression. No arithmetic ZIP range is used. Passing the structure check does not confirm that a prefix belongs to the stated city or state.

In [33]:
geography_entities = {
    "Customers": (raw_datasets["Customers"], "customer_id", "customer_state", "customer_zip_code_prefix"),
    "Sellers": (raw_datasets["Sellers"], "seller_id", "seller_state", "seller_zip_code_prefix"),
}
for dataset_name, (dataframe, id_column, state_column, zip_column) in geography_entities.items():
    record_series_rule(dataset_name, dataframe, state_column, "Recognized Brazilian two-letter state code", ~dataframe[state_column].isin(BRAZIL_STATE_CODES), "Invalid", "Validated against all 27 official UF abbreviations.", "Invalid regions corrupt geographic categories and subgroup metrics.", "Investigate upstream code; do not coerce to a nearby state.", [id_column, state_column, zip_column])
    raw_zip = load_raw_text_csv(DATASET_FILES[dataset_name])[zip_column]
    record_rule(dataset_name, dataframe, zip_column, "Raw ZIP prefix contains exactly five digits", raw_zip.ne(""), ~raw_zip.str.fullmatch(r"[0-9]{5}", na=False), "Invalid", "Literal-text check preserves leading zeros and tests structure, not numeric range.", "Malformed prefixes break geographic lookup and route segmentation.", "Load as fixed-width text in later preparation; investigate malformed source values.", [id_column, zip_column])
    record_series_rule(dataset_name, dataframe, id_column, "Exactly 32 lowercase hexadecimal characters", ~dataframe[id_column].astype("string").str.fullmatch(HEX_ID_PATTERN, na=False), "Invalid", "Objective source identifier representation.", "Malformed entity keys can break item/order relationships.", "Trace malformed identifiers to source; do not guess replacements.", [id_column])

customer_seller_validation = pd.DataFrame(invalid_value_findings_rows[-6:])
display(customer_seller_validation)

,Dataset,Column,Validation Rule,Applicable Records,Affected Records,Affected Percentage,Classification,Evidence,ML Impact,Recommended Action
0,olist_customers_dataset.csv,customer_state,Recognized Brazilian two-letter state code,99441,0,0.0000,Valid,Validated against all 27 official UF abbreviations.,Invalid regions corrupt geographic categories and subgroup metrics.,Investigate upstream code; do not coerce to a nearby state.
1,olist_customers_dataset.csv,customer_zip_code_prefix,Raw ZIP prefix contains exactly five digits,99441,0,0.0000,Valid,"Literal-text check preserves leading zeros and tests structure, not numeric range.",Malformed prefixes break geographic lookup and route segmentation.,Load as fixed-width text in later preparation; investigate malformed source values.
2,olist_customers_dataset.csv,customer_id,Exactly 32 lowercase hexadecimal characters,99441,0,0.0000,Valid,Objective source identifier representation.,Malformed entity keys can break item/order relationships.,Trace malformed identifiers to source; do not guess replacements.
3,olist_sellers_dataset.csv,seller_state,Recognized Brazilian two-letter state code,3095,0,0.0000,Valid,Validated against all 27 official UF abbreviations.,Invalid regions corrupt geographic categories and subgroup metrics.,Investigate upstream code; do not coerce to a nearby state.
4,olist_sellers_dataset.csv,seller_zip_code_prefix,Raw ZIP prefix contains exactly five digits,3095,0,0.0000,Valid,"Literal-text check preserves leading zeros and tests structure, not numeric range.",Malformed prefixes break geographic lookup and route segmentation.,Load as fixed-width text in later preparation; investigate malformed source values.
5,olist_sellers_dataset.csv,seller_id,Exactly 32 lowercase hexadecimal characters,3095,0,0.0000,Valid,Objective source identifier representation.,Malformed entity keys can break item/order relationships.,Trace malformed identifiers to source; do not guess replacements.


### 14.7 Payments Validation

**What are we checking?** Negative and zero payment values/installments, positive payment sequence numbers, and the documented payment-type vocabulary.

**Why does it matter?** Invalid financial values can distort order summaries, while unexpected categories can break later preprocessing.

**How should the result be interpreted?** Negative values/counts and non-positive sequences are invalid. Zero value and zero installments are ambiguous source edge cases and need investigation. The documented literal `not_defined` is retained as a category; it is not silently converted to missing or corrupt. No order-total reconciliation is performed.

In [34]:
payments_raw = raw_datasets["Payments"]
record_series_rule("Payments", payments_raw, "payment_value", "Payment value must not be negative", payments_raw["payment_value"] < 0, "Invalid", "Negative payment-row amounts contradict the documented measure.", "Invalid amounts distort payment aggregates for both objectives.", "Investigate source before later aggregation; do not clip.", ["order_id", "payment_sequential", "payment_value"])
record_series_rule("Payments", payments_raw, "payment_value", "Zero payment value requires investigation", payments_raw["payment_value"].eq(0), "Needs Investigation", "Zero may reflect voucher or accounting behavior and is not proven corrupt.", "Can distort payment totals or zero-rate comparisons.", "Review associated business context before treatment.", ["order_id", "payment_sequential", "payment_type", "payment_value"])
record_series_rule("Payments", payments_raw, "payment_installments", "Installment count must not be negative", payments_raw["payment_installments"] < 0, "Invalid", "A count cannot be negative.", "Invalid counts distort payment-structure features.", "Investigate source; do not clip silently.", ["order_id", "payment_type", "payment_installments"])
record_series_rule("Payments", payments_raw, "payment_installments", "Zero installments require investigation", payments_raw["payment_installments"].eq(0), "Needs Investigation", "The documented source contains rare zero values whose business meaning is ambiguous.", "Can distort installment summaries if later approved as a feature.", "Review payment method and source semantics before treatment.", ["order_id", "payment_type", "payment_installments"])
record_series_rule("Payments", payments_raw, "payment_sequential", "Payment sequence must be positive", payments_raw["payment_sequential"] <= 0, "Invalid", "A within-order payment sequence is one-based.", "Invalid sequences undermine payment grain.", "Investigate before independent order-level aggregation.", ["order_id", "payment_sequential"])
record_series_rule("Payments", payments_raw, "payment_type", "Documented payment-type vocabulary", ~payments_raw["payment_type"].isin(DOCUMENTED_PAYMENT_TYPES), "Invalid", "The documented vocabulary explicitly includes not_defined as a literal category.", "Unexpected categories can break later encoders; payment availability is still unproven.", "Monitor new categories and validate serving-time availability.", ["order_id", "payment_type"])

payments_validation = pd.DataFrame(invalid_value_findings_rows[-6:])
display(payments_validation)
display(payments_raw.loc[payments_raw["payment_type"].eq("not_defined"), ["order_id", "payment_sequential", "payment_type", "payment_installments", "payment_value"]].head(10))

,Dataset,Column,Validation Rule,Applicable Records,Affected Records,Affected Percentage,Classification,Evidence,ML Impact,Recommended Action
0,olist_order_payments_dataset.csv,payment_value,Payment value must not be negative,103886,0,0.0000,Valid,Negative payment-row amounts contradict the documented measure.,Invalid amounts distort payment aggregates for both objectives.,Investigate source before later aggregation; do not clip.
1,olist_order_payments_dataset.csv,payment_value,Zero payment value requires investigation,103886,9,0.0087,Needs Investigation,Zero may reflect voucher or accounting behavior and is not proven corrupt.,Can distort payment totals or zero-rate comparisons.,Review associated business context before treatment.
2,olist_order_payments_dataset.csv,payment_installments,Installment count must not be negative,103886,0,0.0000,Valid,A count cannot be negative.,Invalid counts distort payment-structure features.,Investigate source; do not clip silently.
3,olist_order_payments_dataset.csv,payment_installments,Zero installments require investigation,103886,2,0.0019,Needs Investigation,The documented source contains rare zero values whose business meaning is ambiguous.,Can distort installment summaries if later approved as a feature.,Review payment method and source semantics before treatment.
4,olist_order_payments_dataset.csv,payment_sequential,Payment sequence must be positive,103886,0,0.0000,Valid,A within-order payment sequence is one-based.,Invalid sequences undermine payment grain.,Investigate before independent order-level aggregation.
5,olist_order_payments_dataset.csv,payment_type,Documented payment-type vocabulary,103886,0,0.0000,Valid,The documented vocabulary explicitly includes not_defined as a literal category.,Unexpected categories can break later encoders; payment availability is still unproven.,Monitor new categories and validate serving-time availability.


,order_id,payment_sequential,payment_type,payment_installments,payment_value
51280,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.0000
57411,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.0000
94427,c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.0000


### 14.8 Reviews Validation

**What are we checking?** The documented 1–5 score scale and whether present raw comment text contains a non-whitespace character.

**Why does it matter?** Out-of-range scores are invalid ordinal outcomes. Optional comments, however, are legitimately absent and must not be judged by subjective writing-quality rules.

**How should the result be interpreted?** Empty raw fields are missingness from Task 1 and are excluded from this task's denominator. Whitespace-only present text is structurally malformed. All reviews are retrospective post-purchase information and remain prohibited as purchase-time predictors even if valid.

In [35]:
reviews_raw = raw_datasets["Reviews"]
record_series_rule("Reviews", reviews_raw, "review_score", "Review score must be in the documented 1–5 scale", ~reviews_raw["review_score"].isin([1, 2, 3, 4, 5]), "Invalid", "The source documentation defines a five-point ordinal scale.", "Invalid scores undermine retrospective outcome analysis; review score must never be a purchase-time feature.", "Investigate source and exclude from score analysis only under a documented policy.", ["review_id", "order_id", "review_score"])
review_text_raw = load_raw_text_csv(DATASET_FILES["Reviews"])
for column in ["review_comment_title", "review_comment_message"]:
    present_raw = review_text_raw[column].ne("")
    record_rule("Reviews", reviews_raw, column, "Present review text must not be whitespace-only", present_raw, review_text_raw[column].str.strip().eq(""), "Invalid", "Empty optional comments are missing; whitespace-only present text is structurally empty.", "Only retrospective text analysis is affected; review text is prohibited at purchase time.", "Preserve legitimate absence and investigate only whitespace-only source text.", ["review_id", "order_id", column])

reviews_validation = pd.DataFrame(invalid_value_findings_rows[-3:])
display(reviews_validation)

,Dataset,Column,Validation Rule,Applicable Records,Affected Records,Affected Percentage,Classification,Evidence,ML Impact,Recommended Action
0,olist_order_reviews_dataset.csv,review_score,Review score must be in the documented 1–5 scale,99224,0,0.0000,Valid,The source documentation defines a five-point ordinal scale.,Invalid scores undermine retrospective outcome analysis; review score must never be a purchase-t...,Investigate source and exclude from score analysis only under a documented policy.
1,olist_order_reviews_dataset.csv,review_comment_title,Present review text must not be whitespace-only,11568,2,0.0173,Invalid,Empty optional comments are missing; whitespace-only present text is structurally empty.,Only retrospective text analysis is affected; review text is prohibited at purchase time.,Preserve legitimate absence and investigate only whitespace-only source text.
2,olist_order_reviews_dataset.csv,review_comment_message,Present review text must not be whitespace-only,40977,27,0.0659,Invalid,Empty optional comments are missing; whitespace-only present text is structurally empty.,Only retrospective text analysis is affected; review text is prohibited at purchase time.,Preserve legitimate absence and investigate only whitespace-only source text.


### 14.9 Geolocation Validation

**What are we checking?** Finite coordinates, global latitude/longitude bounds, raw five-digit ZIP structure, recognized states, and a broad Brazil bounding-box screen.

**Why does it matter?** Impossible coordinates can invalidate distance calculations. Malformed ZIP/state codes can corrupt lookup and subgroup logic.

**How should the result be interpreted?** Non-finite or globally out-of-range coordinates are invalid. The approximate box (latitude −34 to 6, longitude −74 to −32) is deliberately labeled **Needs Investigation**: border geometry, islands, source noise, or coordinate mistakes can all produce a screen failure. It is not an automatic deletion rule.

In [36]:
geolocation_raw = raw_datasets["Geolocation"]
latitude = geolocation_raw["geolocation_lat"]
longitude = geolocation_raw["geolocation_lng"]
finite_latitude = latitude.map(lambda value: pd.notna(value) and bool(pd.api.types.is_number(value)) and value not in [float("inf"), float("-inf")])
finite_longitude = longitude.map(lambda value: pd.notna(value) and bool(pd.api.types.is_number(value)) and value not in [float("inf"), float("-inf")])
record_series_rule("Geolocation", geolocation_raw, "geolocation_lat", "Latitude must be finite", ~finite_latitude, "Invalid", "Coordinate arithmetic requires finite numeric values.", "Non-finite latitude breaks distance and regional features.", "Quarantine and trace source; never impute without a later policy.", ["geolocation_zip_code_prefix", "geolocation_lat", "geolocation_lng", "geolocation_state"])
record_series_rule("Geolocation", geolocation_raw, "geolocation_lng", "Longitude must be finite", ~finite_longitude, "Invalid", "Coordinate arithmetic requires finite numeric values.", "Non-finite longitude breaks distance and regional features.", "Quarantine and trace source; never impute without a later policy.", ["geolocation_zip_code_prefix", "geolocation_lat", "geolocation_lng", "geolocation_state"])
record_series_rule("Geolocation", geolocation_raw, "geolocation_lat", "Latitude must be between -90 and 90", ~latitude.between(-90, 90, inclusive="both"), "Invalid", "Global latitude constraint.", "Impossible latitude corrupts geographic distance features.", "Investigate source and exclude only under a documented later policy.", ["geolocation_zip_code_prefix", "geolocation_lat", "geolocation_lng", "geolocation_state"])
record_series_rule("Geolocation", geolocation_raw, "geolocation_lng", "Longitude must be between -180 and 180", ~longitude.between(-180, 180, inclusive="both"), "Invalid", "Global longitude constraint.", "Impossible longitude corrupts geographic distance features.", "Investigate source and exclude only under a documented later policy.", ["geolocation_zip_code_prefix", "geolocation_lat", "geolocation_lng", "geolocation_state"])
brazil_screen_failure = ~(latitude.between(-34, 6, inclusive="both") & longitude.between(-74, -32, inclusive="both"))
coordinate_applicable = latitude.notna() & longitude.notna() & finite_latitude & finite_longitude
record_rule("Geolocation", geolocation_raw, "geolocation_lat, geolocation_lng", "Outside approximate Brazil bounding box", coordinate_applicable, brazil_screen_failure, "Needs Investigation", "Broad screening heuristic only; not an exact political boundary or proof of error.", "Suspicious coordinates can distort route distance and geographic subgroup comparisons.", "Inspect affected ZIP/locality records; do not remove automatically.", ["geolocation_zip_code_prefix", "geolocation_lat", "geolocation_lng", "geolocation_city", "geolocation_state"])
geolocation_zip_raw = load_raw_text_csv(DATASET_FILES["Geolocation"])["geolocation_zip_code_prefix"]
record_rule("Geolocation", geolocation_raw, "geolocation_zip_code_prefix", "Raw ZIP prefix contains exactly five digits", geolocation_zip_raw.ne(""), ~geolocation_zip_raw.str.fullmatch(r"[0-9]{5}", na=False), "Invalid", "Literal-text validation preserves leading zeros; no numeric range is assumed.", "Malformed prefixes break static geographic lookup.", "Load as fixed-width text and trace malformed values.", ["geolocation_zip_code_prefix", "geolocation_state"])
record_series_rule("Geolocation", geolocation_raw, "geolocation_state", "Recognized Brazilian two-letter state code", ~geolocation_raw["geolocation_state"].isin(BRAZIL_STATE_CODES), "Invalid", "Validated against all 27 official UF abbreviations.", "Invalid state categories corrupt geographic grouping.", "Investigate upstream code; do not coerce silently.", ["geolocation_zip_code_prefix", "geolocation_state"])

geolocation_validation = pd.DataFrame(invalid_value_findings_rows[-7:])
display(geolocation_validation)
if invalid_value_samples:
    display(pd.concat(invalid_value_samples, ignore_index=True, sort=False).tail(10))

,Dataset,Column,Validation Rule,Applicable Records,Affected Records,Affected Percentage,Classification,Evidence,ML Impact,Recommended Action
0,olist_geolocation_dataset.csv,geolocation_lat,Latitude must be finite,1000163,0,0.0000,Valid,Coordinate arithmetic requires finite numeric values.,Non-finite latitude breaks distance and regional features.,Quarantine and trace source; never impute without a later policy.
1,olist_geolocation_dataset.csv,geolocation_lng,Longitude must be finite,1000163,0,0.0000,Valid,Coordinate arithmetic requires finite numeric values.,Non-finite longitude breaks distance and regional features.,Quarantine and trace source; never impute without a later policy.
2,olist_geolocation_dataset.csv,geolocation_lat,Latitude must be between -90 and 90,1000163,0,0.0000,Valid,Global latitude constraint.,Impossible latitude corrupts geographic distance features.,Investigate source and exclude only under a documented later policy.
3,olist_geolocation_dataset.csv,geolocation_lng,Longitude must be between -180 and 180,1000163,0,0.0000,Valid,Global longitude constraint.,Impossible longitude corrupts geographic distance features.,Investigate source and exclude only under a documented later policy.
4,olist_geolocation_dataset.csv,"geolocation_lat, geolocation_lng",Outside approximate Brazil bounding box,1000163,31,0.0031,Needs Investigation,Broad screening heuristic only; not an exact political boundary or proof of error.,Suspicious coordinates can distort route distance and geographic subgroup comparisons.,Inspect affected ZIP/locality records; do not remove automatically.
5,olist_geolocation_dataset.csv,geolocation_zip_code_prefix,Raw ZIP prefix contains exactly five digits,1000163,0,0.0000,Valid,Literal-text validation preserves leading zeros; no numeric range is assumed.,Malformed prefixes break static geographic lookup.,Load as fixed-width text and trace malformed values.
6,olist_geolocation_dataset.csv,geolocation_state,Recognized Brazilian two-letter state code,1000163,0,0.0000,Valid,Validated against all 27 official UF abbreviations.,Invalid state categories corrupt geographic grouping.,Investigate upstream code; do not coerce silently.


,Dataset,Rule,order_id,order_item_id,freight_value,product_id,product_weight_g,payment_sequential,payment_type,payment_value,payment_installments,review_id,review_comment_title,review_comment_message,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
18,Reviews,Present review text must not be whitespace-only,d9064a51ed213dcd0cfe59a9039ec458,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,007c08c01178c42ff829560cc24dd0d6,NaN,\r\n\r\n\r\n\r\n\r\n\r\n\r\n\r\n\r\n\r\n,NaN,NaN,NaN,NaN,NaN
19,Reviews,Present review text must not be whitespace-only,81aa3cffc965f29788470b3e2550da67,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1fe9d2ee706fb28c431aa5d933245d45,NaN,\r\n,NaN,NaN,NaN,NaN,NaN
20,Reviews,Present review text must not be whitespace-only,d4c1356b86dcc8a86c603174ba27d9bc,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,479455dcf5c7b17c5025011776ac844e,NaN,\r\n\r\n,NaN,NaN,NaN,NaN,NaN
21,Reviews,Present review text must not be whitespace-only,7a3f8080c604ded6519cd5b601f690b7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1ce11bd2d9c397e0b673035dcc533f5c,NaN,\r\n,NaN,NaN,NaN,NaN,NaN
22,Reviews,Present review text must not be whitespace-only,cf65e1cf310d6860d96e2efaab1f5d75,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,b905a83cd40f67b08363261fc2d2dadb,NaN,\r\n,NaN,NaN,NaN,NaN,NaN
23,Geolocation,Outside approximate Brazil bounding box,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,18243.0000,28.0090,-15.5369,bom retiro da esperanca,SP
24,Geolocation,Outside approximate Brazil bounding box,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,28165.0000,41.6141,-8.4117,vila nova de campos,RJ
25,Geolocation,Outside approximate Brazil bounding box,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,28155.0000,-34.5864,-58.7321,santa maria,RJ
26,Geolocation,Outside approximate Brazil bounding box,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,28155.0000,42.4393,13.8202,santa maria,RJ
27,Geolocation,Outside approximate Brazil bounding box,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,28333.0000,38.3817,-6.3282,raposo,RJ


### 14.10 Category Translation Validation

**What are we checking?** Whether present Portuguese and English category labels are nonblank and whether they follow the source's observed lowercase `snake_case` representation.

**Why does it matter?** Blank values cannot identify a category, while format drift can fragment categorical encoding.

**How should the result be interpreted?** Whitespace-only labels are structurally invalid. The `snake_case` check is only a format heuristic and therefore produces **Needs Investigation**, not a fixed-vocabulary rejection. Key duplication was handled in Task 2, and product-to-translation referential integrity is deliberately out of scope.

In [37]:
translation_raw = raw_datasets["Category Translation"]
for column in ["product_category_name", "product_category_name_english"]:
    text = translation_raw[column].astype("string")
    record_series_rule("Category Translation", translation_raw, column, "Present category label must not be whitespace-only", text.str.strip().eq(""), "Invalid", "Whitespace-only text is structurally empty; nulls are excluded.", "Blank labels erase or fragment category representation.", "Investigate upstream text separately from missingness.", ["product_category_name", "product_category_name_english"])
    record_series_rule("Category Translation", translation_raw, column, "Category label follows observed lowercase snake_case structure", ~text.str.fullmatch(SNAKE_CASE_PATTERN, na=False), "Needs Investigation", "Observed source-format screen, not an invented fixed vocabulary.", "Format drift can split logically related categories during encoding.", "Review unfamiliar formats without rejecting new legitimate categories.", ["product_category_name", "product_category_name_english"])

category_translation_validation = pd.DataFrame(invalid_value_findings_rows[-4:])
display(category_translation_validation)

,Dataset,Column,Validation Rule,Applicable Records,Affected Records,Affected Percentage,Classification,Evidence,ML Impact,Recommended Action
0,product_category_name_translation.csv,product_category_name,Present category label must not be whitespace-only,71,0,0.0000,Valid,Whitespace-only text is structurally empty; nulls are excluded.,Blank labels erase or fragment category representation.,Investigate upstream text separately from missingness.
1,product_category_name_translation.csv,product_category_name,Category label follows observed lowercase snake_case structure,71,0,0.0000,Valid,"Observed source-format screen, not an invented fixed vocabulary.",Format drift can split logically related categories during encoding.,Review unfamiliar formats without rejecting new legitimate categories.
2,product_category_name_translation.csv,product_category_name_english,Present category label must not be whitespace-only,71,0,0.0000,Valid,Whitespace-only text is structurally empty; nulls are excluded.,Blank labels erase or fragment category representation.,Investigate upstream text separately from missingness.
3,product_category_name_translation.csv,product_category_name_english,Category label follows observed lowercase snake_case structure,71,0,0.0000,Valid,"Observed source-format screen, not an invented fixed vocabulary.",Format drift can split logically related categories during encoding.,Review unfamiliar formats without rejecting new legitimate categories.


### 14.11 Consolidated Findings

**What are we checking?** One dynamically calculated table containing every rule, its non-null denominator, affected count/rate, evidence, ML risk, and recommended future action.

**Why does it matter?** Counts without denominators can exaggerate small issues, and a long list of passing checks can obscure the few observations needing attention.

**How should the result be interpreted?** `Valid` means zero failures were observed for that rule in this snapshot. It does not certify factual correctness. Affected percentages use `Applicable Records`, never total rows when nulls were excluded. Summary tables count rules, not source records.

In [38]:
invalid_value_findings = pd.DataFrame(invalid_value_findings_rows)
invalid_value_findings = invalid_value_findings.sort_values(
    ["Dataset", "Classification", "Column", "Validation Rule"]
).reset_index(drop=True)
display(invalid_value_findings)

findings_by_dataset_and_classification = (
    invalid_value_findings.groupby(["Dataset", "Classification"], dropna=False)
    .agg(
        Rules=("Validation Rule", "count"),
        Affected_Records_Across_Rules=("Affected Records", "sum"),
    )
    .reset_index()
)
display(findings_by_dataset_and_classification)

affected_rule_summary = invalid_value_findings.loc[
    invalid_value_findings["Affected Records"] > 0,
    ["Dataset", "Column", "Validation Rule", "Affected Records", "Affected Percentage", "Classification"],
]
display(affected_rule_summary)

,Dataset,Column,Validation Rule,Applicable Records,Affected Records,Affected Percentage,Classification,Evidence,ML Impact,Recommended Action
0,olist_customers_dataset.csv,customer_id,Exactly 32 lowercase hexadecimal characters,99441,0,0.0000,Valid,Objective source identifier representation.,Malformed entity keys can break item/order relationships.,Trace malformed identifiers to source; do not guess replacements.
1,olist_customers_dataset.csv,customer_state,Recognized Brazilian two-letter state code,99441,0,0.0000,Valid,Validated against all 27 official UF abbreviations.,Invalid regions corrupt geographic categories and subgroup metrics.,Investigate upstream code; do not coerce to a nearby state.
2,olist_customers_dataset.csv,customer_zip_code_prefix,Raw ZIP prefix contains exactly five digits,99441,0,0.0000,Valid,"Literal-text check preserves leading zeros and tests structure, not numeric range.",Malformed prefixes break geographic lookup and route segmentation.,Load as fixed-width text in later preparation; investigate malformed source values.
3,olist_geolocation_dataset.csv,"geolocation_lat, geolocation_lng",Outside approximate Brazil bounding box,1000163,31,0.0031,Needs Investigation,Broad screening heuristic only; not an exact political boundary or proof of error.,Suspicious coordinates can distort route distance and geographic subgroup comparisons.,Inspect affected ZIP/locality records; do not remove automatically.
4,olist_geolocation_dataset.csv,geolocation_lat,Latitude must be between -90 and 90,1000163,0,0.0000,Valid,Global latitude constraint.,Impossible latitude corrupts geographic distance features.,Investigate source and exclude only under a documented later policy.
5,olist_geolocation_dataset.csv,geolocation_lat,Latitude must be finite,1000163,0,0.0000,Valid,Coordinate arithmetic requires finite numeric values.,Non-finite latitude breaks distance and regional features.,Quarantine and trace source; never impute without a later policy.
6,olist_geolocation_dataset.csv,geolocation_lng,Longitude must be between -180 and 180,1000163,0,0.0000,Valid,Global longitude constraint.,Impossible longitude corrupts geographic distance features.,Investigate source and exclude only under a documented later policy.
7,olist_geolocation_dataset.csv,geolocation_lng,Longitude must be finite,1000163,0,0.0000,Valid,Coordinate arithmetic requires finite numeric values.,Non-finite longitude breaks distance and regional features.,Quarantine and trace source; never impute without a later policy.
8,olist_geolocation_dataset.csv,geolocation_state,Recognized Brazilian two-letter state code,1000163,0,0.0000,Valid,Validated against all 27 official UF abbreviations.,Invalid state categories corrupt geographic grouping.,Investigate upstream code; do not coerce silently.
9,olist_geolocation_dataset.csv,geolocation_zip_code_prefix,Raw ZIP prefix contains exactly five digits,1000163,0,0.0000,Valid,Literal-text validation preserves leading zeros; no numeric range is assumed.,Malformed prefixes break static geographic lookup.,Load as fixed-width text and trace malformed values.


,Dataset,Classification,Rules,Affected_Records_Across_Rules
0,olist_customers_dataset.csv,Valid,3,0
1,olist_geolocation_dataset.csv,Needs Investigation,1,31
2,olist_geolocation_dataset.csv,Valid,6,0
3,olist_order_items_dataset.csv,Needs Investigation,1,383
4,olist_order_items_dataset.csv,Valid,7,0
5,olist_order_payments_dataset.csv,Needs Investigation,2,11
6,olist_order_payments_dataset.csv,Valid,4,0
7,olist_order_reviews_dataset.csv,Invalid,2,29
8,olist_order_reviews_dataset.csv,Valid,1,0
9,olist_orders_dataset.csv,Valid,3,0


,Dataset,Column,Validation Rule,Affected Records,Affected Percentage,Classification
3,olist_geolocation_dataset.csv,"geolocation_lat, geolocation_lng",Outside approximate Brazil bounding box,31,0.0031,Needs Investigation
10,olist_order_items_dataset.csv,freight_value,Zero freight is a plausible free-shipping case,383,0.3400,Needs Investigation
18,olist_order_payments_dataset.csv,payment_installments,Zero installments require investigation,2,0.0019,Needs Investigation
19,olist_order_payments_dataset.csv,payment_value,Zero payment value requires investigation,9,0.0087,Needs Investigation
24,olist_order_reviews_dataset.csv,review_comment_message,Present review text must not be whitespace-only,27,0.0659,Invalid
25,olist_order_reviews_dataset.csv,review_comment_title,Present review text must not be whitespace-only,2,0.0173,Invalid
30,olist_products_dataset.csv,product_weight_g,Zero physical measurement requires investigation,4,0.0121,Needs Investigation


### 14.12 ML Impact Assessment

| Quality issue | Classification impact | Regression impact | Future preparation consideration |
|---|---|---|---|
| Invalid price, freight, or payment values | Can create misleading risk differences across value/freight segments | Can bias duration estimates through corrupted numeric distributions | Trace source; aggregate only after a documented handling decision |
| Invalid product weight or dimensions | Can make shipment-complexity signals unreliable | Can produce physically implausible relationships with delivery days | Preserve issue flags; never silently clip or treat null as invalid |
| Malformed ZIP/state values | Can misstate regional late-rate comparisons and subgroup performance | Can corrupt location/distance relationships with delivery duration | Preserve ZIP as five-character text and keep unmatched/invalid indicators |
| Impossible or Brazil-screened coordinates | Can assign orders to misleading geographic risk patterns | Can severely distort route-distance predictors | Separate global invalidity from the non-authoritative Brazil heuristic |
| Unexpected categorical values | Can break fitted encoders or silently form unstable groups | Same preprocessing and stability risk | Monitor controlled vocabularies; allow reviewed evolution where appropriate |
| Invalid review scores/text | Affects retrospective analysis only | Affects retrospective analysis only | Reviews occur after purchase and remain excluded from both models |

Business-rule validity, prediction-time availability, and feature usefulness are three separate gates. A valid field is not automatically eligible or useful. In particular, review fields and later order status are unavailable at `order_purchase_timestamp`; actual delivery is target-only. Candidate item, product, seller, customer, geolocation, and payment fields still require an inference-availability contract, duplicate-safe order-level aggregation, and training-only preprocessing. No feature is implemented or approved in this section.

### 14.13 Learning Questions

These questions are intentionally unanswered.

1. What is the difference between an invalid value and a missing value?
2. Why can a numeric value have a valid dtype but still be invalid?
3. Why should zero prices and zero freight values not automatically be removed?
4. What is the difference between a domain constraint and a suspicious-value heuristic?
5. Why do ZIP codes require structural rather than numeric range validation?
6. Why can geographically implausible coordinates be suspicious without being conclusively invalid?
7. How can invalid values affect classification and regression differently?
8. Why should we document invalid values before deciding how to clean them?